# Quedas e aumentos bruscos de matrículas (INEP 2011–2025)

**Ideia do notebook:** para cada escola, calcular a variação percentual de matrículas **em relação ao ano anterior**, e usar boxplots + cerca de Tukey (Q1 − k·AIQ, Q3 + k·AIQ) para definir o que é uma queda (abaixo da cerca inferior) ou um aumento (acima da cerca superior) "bruscos".

**Como usar:**
1. Edite **apenas a célula "PARÂMETROS"** (etapa 1).
2. Rode *Run All*. No fim, abra o arquivo **`relatorio_quedas_bruscas.html`** (etapa 11): ele junta **todos os gráficos e tabelas** de todos os cenários (ex.: k=1,5 e k=2) em uma única página.
3. Para acumular mais cenários (outros k, quartis, critérios, etapas, filtros), use a seção **10) CENÁRIOS** da célula de parâmetros.

| Etapa | O que faz |
|---|---|
| 0 | Imports |
| 1 | **PARÂMETROS (único lugar para editar)** |
| 2 | Carrega a base e monta a variação % ano a ano |
| 3 | Calcula limites (por ano e global) e marca as quedas bruscas |
| 4 | Boxplot por ano (limite por ano × limite global) |
| 5 | Proporção de quedas e aumentos bruscos por ano |
| 6 | Cruzamento com rede, localização, região, UF e modalidade |
| 6.2b | **Boxplots lado a lado em cada ano (rural × urbana, rede, região...)** |
| 7 | Sensibilidade: vários quartis × vários multiplicadores |
| 8 | Comparação entre etapas (AI, AF, EM, total) |
| 9 | Exportação |
| 10 | **Cenários extras** (vários k / quartis / critérios / filtros) + comparação entre cenários |
| 11 | **Relatório HTML** com todos os gráficos |

## Etapa 0 — Imports

In [136]:
import io
import json
import time
import zipfile
import datetime
import html as _html
from contextlib import contextmanager
from itertools import product
from pathlib import Path

import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()   # evita o limite de 5000 linhas do Altair


def estilizar(chart):
    """Visual limpo e consistente para todos os gráficos."""
    return (chart
            .configure_view(stroke=None)
            .configure_axis(gridColor="#E5E7EB", domainColor="#9CA3AF", tickColor="#9CA3AF",
                            labelFontSize=12, titleFontSize=13, labelColor="#374151", titleColor="#111827")
            .configure_title(fontSize=globals().get("FONTE_TITULO", 17), subtitleFontSize=globals().get("FONTE_SUBTITULO", 12),
                 subtitleColor="#6B7280", anchor="start", color="#111827")
            .configure_legend(labelFontSize=12, titleFontSize=12, symbolSize=120, labelLimit=400)
            .configure_range(category={"scheme": "set2"}))


pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:,.2f}".format)

# ============================================================
#  COLETOR DO RELATÓRIO HTML
#  Tudo que passa por mostrar() / mostrar_tabela() / mostrar_texto()
#  aparece no notebook E é guardado para o relatório HTML (etapa 11).
# ============================================================
try:
    display
except NameError:                       # rodando fora do Jupyter
    def display(x):
        print(x)

RELATORIO = []                  # itens guardados para o relatório
CENARIO_ATUAL = "Principal"     # rótulo do cenário em execução (muda sozinho na etapa 10)
SECAO_ATUAL = "Geral"


def _exibir():
    return globals().get("EXIBIR_NO_NOTEBOOK", True)


def secao(nome):
    """Define a seção (título) sob a qual os próximos itens entram no relatório."""
    global SECAO_ATUAL
    SECAO_ATUAL = nome


_N_IMAGEM = [0]
_CEN_BASE = [None]


def _slug(txt):
    """Transforma um título em nome de arquivo seguro."""
    import re, unicodedata
    t = unicodedata.normalize("NFKD", str(txt)).encode("ascii", "ignore").decode()
    return re.sub(r"[^A-Za-z0-9]+", "_", t).strip("_")[:70] or "grafico"


def _salvar_imagem(chart, titulo):
    """Se SALVAR_IMAGENS = True, salva o gráfico em PASTA_IMAGENS (precisa de vl-convert-python)."""
    g = globals()
    if not g.get("SALVAR_IMAGENS", False):
        return
    try:
        pasta = Path(g["PASTA_IMAGENS"])
        pasta.mkdir(parents=True, exist_ok=True)
        _N_IMAGEM[0] += 1
        nome = f"{_N_IMAGEM[0]:02d}_{_slug(titulo)[:50]}"
        if _CEN_BASE[0] is None:
            _CEN_BASE[0] = CENARIO_ATUAL
        if CENARIO_ATUAL != _CEN_BASE[0]:          # só distingue cenários extras
            nome += "_cen" + _slug(CENARIO_ATUAL)[:25]
        for fmt in g.get("FORMATOS_IMAGEM", ["png"]):
            destino = pasta / f"{nome}.{fmt}"
            if fmt == "png":
                chart.save(str(destino), scale_factor=g.get("ESCALA_PNG", 3))
            else:
                chart.save(str(destino))
    except Exception as e:
        print(f"⚠ Não consegui salvar a imagem '{titulo}': {e}\n"
              "  Instale com:  %pip install vl-convert-python   (e reinicie o kernel)")


# ============================================================
#  TRADUÇÃO DOS GRÁFICOS (português -> inglês)
#  Tudo que aparece nos gráficos (títulos, eixos, legendas, tooltips e categorias) passa por traduzir_grafico(),
#  chamada dentro de mostrar(). Para mudar uma palavra, edite os dicionários abaixo.
#  IDIOMA_GRAFICOS = "pt" (célula de parâmetros) desliga a tradução. Tabelas, textos impressos e o relatório HTML não são traduzidos.
# ============================================================
TRADUCAO_VALORES = {          # categorias que vêm dos dados (valor exato -> tradução)
    "Pública": "Public", "Privada": "Private", "Federal": "Federal", "Estadual": "State", "Municipal": "Municipal",
    "Urbana": "Urban", "Rural": "Rural",
    "Norte": "North", "Nordeste": "Northeast", "Sudeste": "Southeast", "Sul": "South", "Centro-Oeste": "Central-West",
    "AI": "Early elementary (AI)", "AF": "Late elementary (AF)", "EM": "High school (EM)",
    "TOTAL": "Total (AI + AF + EM)", "(vazio)": "(missing)",
    "queda": "drop", "aumento": "increase", "queda brusca": "sharp drop", "aumento brusco": "sharp increase",
    "limite por ano": "yearly limit", "limite global": "global limit", "por ano": "yearly",
    "cerca inferior": "lower fence", "cerca superior": "upper fence",
}
TRADUCAO_EXATA = {            # títulos/rótulos inteiros (texto exato -> tradução)
    "REDE": "Network", "DEP_ADM": "Administrative type", "LOCALIZACAO": "Location", "REGIAO": "Region",
    "UF": "State", "ETAPA_ANALISADA": "Stage", "Ano": "Year", "Tipo": "Type",
    "Anos iniciais (AI)": "Early elementary (AI)", "Anos finais (AF)": "Late elementary (AF)",
    "Ensino médio (EM)": "High school (EM)", "Total (AI + AF + EM)": "Total (AI + AF + EM)",
    "AI": "Early elementary (AI)", "AF": "Late elementary (AF)", "EM": "High school (EM)", "TOTAL": "Total (AI + AF + EM)",
    "escolas": "schools", "limite inferior": "lower limit", "limite superior": "upper limit", "mediana": "median",
    "quartil inferior": "lower quartile", "quartil superior": "upper quartile", "% marcadas": "% flagged",
    "cenário": "scenario", "limite (variação %)": "limit (change %)", "multiplicador k": "multiplier k",
    "quartis (inf/sup)": "quartiles (lower/upper)",
}
TRADUCAO_FRASES = [           # (expressão regular, substituição), aplicadas em ordem nos textos dos gráficos
    (r"% de escolas com queda \(tracejada\) e aumento \(contínua\) brusco por (\S+) e por etapa",
     r"% of schools with a sharp drop or increase, by \1 and stage"),
    (r"% de escolas com queda \(tracejada\) e aumento \(contínua\) brusco por",
     "% of schools with a sharp drop (dashed) and sharp increase (solid) by"),
    (r"% de escolas com variação brusca \(queda ou aumento\) por", "% of schools with a sharp change (drop or increase) by"),
    (r"% de escolas com variação brusca \(queda ou aumento\)", "% of schools with a sharp change (drop or increase)"),
    (r"% de escolas com queda e com aumento brusco, por cenário", "% of schools with a sharp drop and a sharp increase, by scenario"),
    (r"% com variação brusca \(queda ou aumento\)", "% with a sharp change (drop or increase)"),
    (r"% de escolas marcadas", "% of flagged schools"),
    (r"% de escolas", "% of schools"),
    (r"Proporção de escolas com queda e aumento brusco por ano", "Share of schools with a sharp drop and a sharp increase, by year"),
    (r"Cercas de Tukey por ano, por cenário", "Tukey fences by year, by scenario"),
    (r"Sensibilidade do critério", "Criterion sensitivity"),
    (r"Variação % de matrículas vs\. ano anterior", "Change in enrollment vs. previous year (%)"),
    (r"Boxplot de (\S+) em cada ano — cercas gerais \(todas as escolas do ano\)",
     r"Boxplot of \1 in each year — general fences (all schools in the year)"),
    (r"Boxplot de (\S+) em cada ano — cercas próprias de cada categoria",
     r"Boxplot of \1 in each year — each category's own fences"),
    (r"Boxplot por", "Boxplot by"),
    (r"\(limite ano\)", "(yearly limit)"),
    (r"\(limite global\)", "(global limit)"),
    (r"Limite POR ANO —", "Yearly limit —"),
    (r"Limite GLOBAL —", "Global limit —"),
    # subtítulos
    (r"Caixa = quartis", "Box = quartiles"),
    (r"traço branco = mediana", "white tick = median"),
    (r"bigodes = dados dentro da cerca", "whiskers = data within the fence"),
    (r"traço vermelho = limite inferior", "red tick = lower limit"),
    (r"traço laranja = limite superior", "orange tick = upper limit"),
    (r"traço vermelho/laranja = cerca inferior/superior", "red/orange tick = lower/upper fence"),
    (r"roxo = anos em foco \(gráfico geral\) / cor = categoria \(facetas\)", "purple = focus years (overall chart) / color = category (facets)"),
    (r"uma cor por categoria", "one color per category"),
    (r"Eixo fixo: (\S+) a (\S+) \(cercas = traços vermelho e laranja; o que passa do eixo fica fora da vista\)",
     r"Fixed axis: \1 to \2 (fences = red and orange ticks; values beyond the axis are out of view)"),
    (r"Eixo ampliado: (\S+) a (\S+) \(o que sai dessa faixa fica fora da vista; os números continuam nas tabelas e tooltips\)",
     r"Zoomed axis: \1 to \2 (values outside this range are out of view; the numbers remain in the tables and tooltips)"),
    (r"Eixo y ampliado: (\S+) a (\S+)", r"Zoomed y-axis: \1 to \2"),
    (r"eixo y ampliado: (\S+) a (\S+)", r"zoomed y-axis: \1 to \2"),
    (r"\bquartis\b", "quartiles"),
    (r"critério=estatistico", "criterion=statistical"),
    (r"critério=absoluto", "criterion=absolute"),
    (r"critério=ambos", "criterion=both"),
    (r"\bestatistico\b", "statistical"),
    (r"\babsoluto\b", "absolute"),
    (r"\bambos\b", "both"),
    (r"\bPrincipal\b", "Main"),
    (r"etapa=", "stage="),
    (r"mín\. (\d+) alunos", r"min. \1 students"),
    # nomes das dimensões dentro de frases
    (r"\bREDE\b", "network"), (r"\bDEP_ADM\b", "administrative type"), (r"\bLOCALIZACAO\b", "location"),
    (r"\bREGIAO\b", "region"), (r"\bUF\b", "state"), (r"\bETAPA_ANALISADA\b", "stage"),
    (r"\betapa\b", "stage"),
    (r"\bAno\b", "Year"),
]
TRADUCAO_CAMPOS = {           # nome da coluna -> rótulo, quando o gráfico não definiu um título (legendas e tooltips)
    "ANO": "Year", "CO_ESCOLA": "School ID", "NO_ESCOLA": "School name", "DEP_ADM": "Administrative type",
    "ETAPA_ANALISADA": "Stage", "LOCALIZACAO": "Location", "REDE": "Network", "REGIAO": "Region", "UF": "State",
    "MARCADAS": "Flagged schools", "QUEDAS": "Flagged schools", "MATRICULAS": "Enrollment",
    "MAT_ANT": "Previous-year enrollment", "N": "Schools", "PCT_QUEDA": "% flagged", "VAR_PCT": "Change (%)",
    "bigode_inf": "Lower whisker", "bigode_sup": "Upper whisker", "cenario": "Scenario", "cerca": "Fence",
    "etapa": "Stage", "k": "k", "lim_inf": "Lower limit", "lim_sup": "Upper limit", "limite": "Limit",
    "mediana": "Median", "pct": "%", "q_inf": "Lower quartile", "q_sup": "Upper quartile", "quantis": "Quartiles",
    "tipo": "Type", "valor": "Value",
}


import re as _re


def _trad_exato(s, tabela):
    return tabela.get(s, s) if isinstance(s, str) else s


def _trad_frase(s):
    """Traduz um título/subtítulo/rótulo: primeiro o texto exato, depois as frases (ver TRADUCAO_FRASES)."""
    if not isinstance(s, str):
        return s
    if s in TRADUCAO_EXATA:
        return TRADUCAO_EXATA[s]
    for padrao, subst in TRADUCAO_FRASES:
        s = _re.sub(padrao, subst, s)
    return s


def _trad_titulo(v):
    if isinstance(v, str):
        return _trad_frase(v)
    if isinstance(v, list):
        return [_trad_titulo(x) for x in v]
    if isinstance(v, dict):
        return _trad_walk(v)
    return v


def _trad_dados(v):
    """Valores das tabelas embutidas no gráfico (categorias como 'Pública', 'queda'...)."""
    if isinstance(v, str):
        if v in TRADUCAO_VALORES:
            return TRADUCAO_VALORES[v]
        if " · " in v:                       # nomes de cenários, ex.: "k=2 · quartis 0.25/0.75 · estatistico"
            for padrao, subst in TRADUCAO_FRASES:
                v = _re.sub(padrao, subst, v)
        return v
    if isinstance(v, list):
        return [_trad_dados(x) for x in v]
    if isinstance(v, dict):
        return {k: _trad_dados(x) for k, x in v.items()}
    return v


# canais posicionais (x, y) ficam de fora: o Vega-Lite junta os títulos das camadas e um título extra viraria "A, B"
_CANAIS = ("color", "fill", "stroke", "strokeDash", "shape", "size", "opacity",
           "xOffset", "yOffset", "row", "column", "facet", "tooltip", "detail", "text", "order")


def _trad_canal(c, campos):
    if isinstance(c, list):
        for x in c:
            _trad_canal(x, campos)
    elif isinstance(c, dict) and "field" in c and "title" not in c and isinstance(c["field"], str):
        if c["field"] in campos:
            c["title"] = campos[c["field"]]


def _trad_walk(o, campos=None):
    campos = campos if campos is not None else TRADUCAO_CAMPOS
    if isinstance(o, list):
        return [_trad_walk(x, campos) for x in o]
    if not isinstance(o, dict):
        return o
    for k in list(o):
        v = o[k]
        if k in ("datasets", "values"):
            o[k] = _trad_dados(v)
        elif k in ("title", "subtitle", "text"):
            o[k] = _trad_titulo(v)
        elif k in ("domain", "sort") and isinstance(v, list):
            o[k] = _trad_dados(v)
        elif k == "encoding" and isinstance(v, dict):
            for canal in _CANAIS:
                if canal in v:
                    _trad_canal(v[canal], campos)
            o[k] = _trad_walk(v, campos)
        else:
            o[k] = _trad_walk(v, campos)
    return o


def traduzir_grafico(chart):
    """Devolve o gráfico com títulos, eixos, legendas, tooltips e categorias em inglês (IDIOMA_GRAFICOS = 'en')."""
    if IDIOMA_GRAFICOS != "en" or chart is None:
        return chart
    try:
        spec = _trad_walk(chart.to_dict())
        return alt.Chart.from_dict(spec)
    except Exception as e:
        print(f"[aviso] não consegui traduzir o gráfico ({e}); mantido no idioma original.")
        return chart


def mostrar(chart, titulo, nota=None):
    """Mostra o gráfico no notebook e o guarda no relatório."""
    if chart is None:
        return
    chart = traduzir_grafico(chart)
    try:
        spec = chart.to_dict()
    except Exception as e:
        spec = None
        print(f"⚠ Não consegui guardar '{titulo}' no relatório: {e}")
    if spec is not None:
        RELATORIO.append({"tipo": "grafico", "cenario": CENARIO_ATUAL, "secao": SECAO_ATUAL,
                          "titulo": titulo, "nota": nota, "spec": spec})
    if _exibir():
        display(chart)
    _salvar_imagem(chart, titulo)


def mostrar_tabela(df, titulo, nota=None, exibir=True):
    """Mostra a tabela no notebook e a guarda no relatório."""
    if df is None:
        return
    if isinstance(df, pd.Series):
        df = df.to_frame()
    tabela = df.to_html(classes="tabela", border=0, na_rep="–", float_format=lambda x: f"{x:,.2f}")
    RELATORIO.append({"tipo": "tabela", "cenario": CENARIO_ATUAL, "secao": SECAO_ATUAL,
                      "titulo": titulo, "nota": nota, "html": tabela})
    if exibir and _exibir():
        display(df)


def mostrar_texto(texto, titulo=None, nivel="info"):
    """Mensagem curta (nivel = 'info' ou 'aviso'): vai para o notebook e para o relatório."""
    RELATORIO.append({"tipo": "texto", "cenario": CENARIO_ATUAL, "secao": SECAO_ATUAL,
                      "titulo": titulo, "nota": None, "texto": texto, "nivel": nivel})
    if _exibir():
        print(("⚠ " if nivel == "aviso" else "") + texto)


def nome_cenario(cfg=None):
    """Rótulo legível do cenário a partir dos parâmetros em vigor."""
    nome = f"k={MULTIPLICADOR:g} · quartis {Q_INFERIOR:.2f}/{Q_SUPERIOR:.2f} · {CRITERIO}"
    fixos = ("MULTIPLICADOR", "Q_INFERIOR", "Q_SUPERIOR", "CRITERIO", "nome")
    extras = [f"{k}={v}" for k, v in (cfg or {}).items() if k not in fixos]
    return nome + (" · " + ", ".join(extras) if extras else "")


PARAMS_RESUMO = ["ETAPA", "INCLUIR_INTEGRAL", "ANO_INICIO", "ANO_FIM", "FILTRO_DEP_ADM", "FILTRO_REDE", "FILTRO_LOCALIZACAO",
                 "FILTRO_REGIAO", "FILTRO_UF", "MATRICULA_MINIMA_ANO_ANTERIOR", "EXCLUIR_FECHAMENTO",
                 "Q_INFERIOR", "Q_SUPERIOR", "MULTIPLICADOR", "CRITERIO", "CORTE_ABSOLUTO_PCT",
                 "CORTE_ABSOLUTO_AUMENTO_PCT", "DIRECAO", "LIMITE_PARA_CRUZAMENTO", "ZOOM_Y", "ZOOM_MARGEM",
                 "Y_MIN", "Y_MAX", "ANOS_FOCO"]


def registrar_parametros():
    g = globals()
    tab = pd.DataFrame({"valor": [str(g.get(k, "—")) for k in PARAMS_RESUMO]}, index=PARAMS_RESUMO)
    tab.index.name = "parâmetro"
    mostrar_tabela(tab, "Parâmetros em vigor neste cenário", exibir=False)

## Etapa 1 — PARÂMETROS (edite só aqui)

In [157]:
# ============================================================
#  PARÂMETROS — TUDO QUE VOCÊ PRECISA EDITAR ESTÁ NESTA CÉLULA
# ============================================================

# ---------- 1) ARQUIVO ----------
RAIZ = Path.cwd()
while not (RAIZ / "dados").exists() and RAIZ != RAIZ.parent:
    RAIZ = RAIZ.parent
CAMINHO_ARQUIVO = RAIZ / "dados" / "bd5.zip"# .zip (com um .csv/.xlsx/.parquet dentro), ou direto .csv, .xlsx, .parquet
NOME_ARQUIVO_NO_ZIP = None   # se o zip tiver vários arquivos, escreva o nome do que quer ler. None = usa o primeiro
CSV_SEP         = ";"        # se der erro de colunas, tente ";"
CSV_ENCODING    = "utf-8"    # se der erro de acentos, tente "latin-1"

# ---------- 2) QUAL "NÚMERO DE MATRÍCULAS" ANALISAR ----------
ETAPA            = "TOTAL"   # "TOTAL" (AI+AF+EM), uma etapa ("AI", "AF" ou "EM")
                             # ou a SOMA de duas etapas: "AI+AF", "AF+EM" ou "AI+EM" (a ordem não importa)
INCLUIR_INTEGRAL = True      # True = regular + integral | False = só regular

# ---------- 3) FILTROS DA AMOSTRA (None = não filtra) ----------
ANO_INICIO   = 2012          # primeiro ano com variação (2011 não tem ano anterior)
ANO_FIM      = 2025
FILTRO_DEP_ADM     = None    # ex.: ["Privada"] ou ["Estadual", "Municipal"]
FILTRO_REDE        = None    # ex.: ["Pública"] ou ["Privada"]. REDE = Pública (Federal + Estadual + Municipal) × Privada
FILTRO_LOCALIZACAO = None    # ex.: ["Urbana"]
FILTRO_REGIAO      = None    # ex.: ["Sudeste", "Sul"]
FILTRO_UF          = None    # ex.: ["RJ", "SP"]

MATRICULA_MINIMA_ANO_ANTERIOR = 30    # ignora escolas muito pequenas (ruído). Use 0 para não filtrar
EXCLUIR_FECHAMENTO = True    # True = tira escolas com 0 matrículas no ano atual (queda de -100%)

# ---------- 4) CRITÉRIO DE "QUEDA BRUSCA" ----------
Q_INFERIOR    = 0.25         # quartil de baixo (0.25 = Q1). Ex.: 0.10 para P10
Q_SUPERIOR    = 0.75         # quartil de cima  (0.75 = Q3). Ex.: 0.90 para P90
MULTIPLICADOR = 1.5       # o "1,5" do 1,5·AIQ. Teste 1.0, 2.0, 2.5, 3.0...

CRITERIO = "estatistico"     # "estatistico" -> abaixo de Q_inf − k·AIQ
                             # "absoluto"    -> variação <= CORTE_ABSOLUTO_PCT
                             # "ambos"       -> precisa cumprir os dois
CORTE_ABSOLUTO_PCT = -30     # QUEDA: variação <= este valor (em %), usado se CRITERIO = "absoluto" ou "ambos"
CORTE_ABSOLUTO_AUMENTO_PCT = 30   # AUMENTO: variação >= este valor (em %), idem

# O mesmo CRITERIO vale para os dois lados:
#   estatistico -> queda abaixo de Q_inf − k·AIQ | aumento acima de Q_sup + k·AIQ

DIRECAO = "ambos"            # o que analisar nas etapas 4, 6, 7, 8 e 9:
                             #   "queda"   -> só quedas bruscas
                             #   "aumento" -> só aumentos bruscos
                             #   "ambos"   -> qualquer variação brusca (queda OU aumento)
LIMITE_PARA_CRUZAMENTO = "ANO"   # qual flag usar nas etapas 6 e 9: "ANO" ou "GLOBAL"

# ---------- 5) GRÁFICOS ----------
ZOOM_Y = "meio"              # eixo y dos boxplots:
                             #   "meio"   -> aproxima da região central (caixas e medianas); cercas e outliers extremos saem da vista
                             #   "cercas" -> mostra também as cercas de Tukey (limites vermelho/laranja)
ZOOM_MARGEM = 0.5            # folga em volta (fração da amplitude das caixas). Menor = mais zoom. Ex.: 0.2 (muito perto) a 1.5 (mais folgado)
Y_MIN, Y_MAX = None, None    # opcional: fixa o eixo y manualmente (ignora ZOOM_Y). Ex.: Y_MIN, Y_MAX = -8, 8
IDIOMA_GRAFICOS = "en"       # "en" = títulos, eixos, legendas, tooltips e categorias dos gráficos em inglês | "pt" = em português
LARGURA_GRAFICO = 1000       # largura dos gráficos (px)
ALTURA_GRAFICO  = 760        # altura dos gráficos (px)
COR_CAIXA       = "#5B8DB8"  # caixa dos anos normais
COR_CAIXA_FOCO  = "#8E6BBF"  # caixa dos ANOS_FOCO (destaca a pandemia)
COR_QUEDA       = "#D1495B"  # limite inferior / quedas
COR_AUMENTO     = "#E8A33D"  # limite superior / aumentos
ANOS_FOCO    = [2019, 2020, 2021, 2022]     # anos para comparar pandemia
MOSTRAR_PONTOS = "marcados"  # pontos no boxplot: "marcados" (conforme DIRECAO), "outliers" (fora da cerca, dos dois lados), "nenhum"
MAX_PONTOS_POR_ANO = 300     # amostra de pontos por ano (evita gráfico pesado)
BOXPLOT_POR = ["REDE", "DEP_ADM", "LOCALIZACAO", "REGIAO"]   # dimensões para boxplot facetado ("REDE" = pública × privada)
BOXPLOT_SO_ANOS_FOCO = True  # True = facetas só com ANOS_FOCO | False = todos os anos
N_MIN_CATEGORIA = 30         # categorias com menos escolas/ano que isso são escondidas nas proporções
BOXPLOT_ANO_ORIENTACAO = "horizontal"   # "horizontal" (anos na vertical) ou "vertical" (formato antigo)
BOXPLOT_ANO_ESCALA     = (-100, 100)    # eixo da variação %. None = usa o zoom de ZOOM_Y / ZOOM_MARGEM
BOXPLOT_ANO_LARGURA    = 1000           # largura (px)
BOXPLOT_ANO_ALTURA     = 1500           # altura (px): maior = mais comprido
FONTE_TITULO    = 24    # título dos gráficos
FONTE_SUBTITULO = 14    # subtítulo (as linhas de legenda dos boxplots)

# ---------- 6) CRUZAMENTOS ----------
# Dimensões usadas nas proporções. A modalidade predominante da etapa (ENS_PRED_AI/AF/EM)
# é adicionada automaticamente quando ETAPA != "TOTAL".
# "REDE" agrupa Federal + Estadual + Municipal como "Pública" e compara com "Privada".
# "DEP_ADM" mantém as quatro dependências separadas. Pode usar as duas ou só uma delas.
DIMENSOES = ["REDE", "DEP_ADM", "LOCALIZACAO", "REGIAO", "UF"]
PERFIL_ANOS = ANOS_FOCO      # anos usados na tabela de perfil das escolas com queda

# ---------- 7) SENSIBILIDADE (grade de testes) ----------
GRADE_MULTIPLICADORES = [1.0, 1.5, 2.0, 2.5, 3.0]
GRADE_QUANTIS = [(0.25, 0.75), (0.10, 0.90), (0.05, 0.95)]   # pares (inferior, superior)

# ---------- 8) COMPARAÇÃO ENTRE ETAPAS ----------
ETAPAS_COMPARAR = ["TOTAL", "AI", "AF", "EM"]   # aceita também somas, ex.: ["AI", "AF", "AI+AF", "TOTAL"]

# Gráfico "queda (tracejada) × aumento (contínua)" com uma etapa embaixo da outra (etapa 8b)
ETAPAS_EMPILHADAS    = ["AI", "AF", "EM"]   # ordem de cima para baixo; aceita também somas, ex.: ["AI+AF", "EM"]
DIM_EMPILHADO        = "REDE"               # "REDE", "DEP_ADM", "LOCALIZACAO" ou "REGIAO" (dimensão com até 8 categorias)
EIXO_Y_COMPARTILHADO = True                 # True = mesma escala do eixo Y nos painéis (comparação justa)
ALTURA_PAINEL        = 220                  # altura de cada painel, em pixels

# ---------- 9) EXPORTAÇÃO ----------
EXPORTAR_CSV = False
(RAIZ / "analise" / "resultados").mkdir(parents=True, exist_ok=True)
ARQUIVO_SAIDA = RAIZ / "analise" / "resultados" / "escolas_queda_brusca.csv"

# ---------- 10) CENÁRIOS: acumular parâmetros no MESMO relatório (etapa 10) ----------
# O cenário "Principal" é o configurado nas seções acima (ex.: MULTIPLICADOR = 1.5).
# Os cenários abaixo rodam depois e entram no mesmo relatório + uma aba de comparação.
# Cada combinação de k × quartis × critério vira um cenário (o que for igual ao Principal é pulado).
CENARIO_MULTIPLICADORES = [1.5, 2.0]        # ex.: [1.5, 2.0, 3.0]. Lista vazia [] = não cria cenários
CENARIO_QUANTIS = [(0.25, 0.75)]            # pares (inferior, superior). Ex.: [(0.25, 0.75), (0.10, 0.90)]
CENARIO_CRITERIOS = ["estatistico"]         # ex.: ["estatistico", "absoluto", "ambos"]

# Cenários livres: qualquer parâmetro desta célula pode ser trocado. "nome" é opcional.
# Ex.: [{"nome": "Só ensino médio", "ETAPA": "EM"},
#       {"nome": "Sudeste, k=2", "FILTRO_REGIAO": ["Sudeste"], "MULTIPLICADOR": 2.0},
#       {"nome": "Só quedas, corte -40%", "DIRECAO": "queda", "CRITERIO": "absoluto", "CORTE_ABSOLUTO_PCT": -40}]
CENARIOS_EXTRA = []
CENARIOS_RODAR_ETAPAS = False   # True = refaz também a etapa 8 (comparação AI/AF/EM/total) em cada cenário (mais lento)

# ---------- 11) COMPARATIVOS: boxplots lado a lado, em cada ano (etapa 6.2b) ----------
COMPARATIVOS = ["LOCALIZACAO","REDE","DEP_ADM","REGIAO",""]  # dimensões comparadas. Ex.: ["LOCALIZACAO", "REDE", "DEP_ADM", "REGIAO"]  (LOCALIZACAO = rural × urbana; REDE = pública × privada)
COMPARATIVO_FILTRO = None       # opcional: restringe categorias. Ex.: {"LOCALIZACAO": ["Rural", "Urbana"]}
COMPARATIVO_ANOS = None         # None = todos os anos. Ex.: [2019, 2020, 2021]
COMPARATIVO_LIMITES = ["geral", "categoria"]   # "geral"     -> cercas calculadas com TODAS as escolas do ano (mesmas das outras etapas)
                                               # "categoria" -> cada categoria (ex.: rural e urbana) com as SUAS cercas
                                               # com os dois, saem dois gráficos por dimensão

# ---------- 13) SALVAR IMAGENS DOS GRÁFICOS (para o pôster) ----------
SALVAR_IMAGENS  = True         # True = salva cada gráfico em PASTA_IMAGENS (precisa: %pip install vl-convert-python)
FORMATOS_IMAGEM = ["pdf"]       # "png", "svg" e/ou "pdf" (svg/pdf são vetoriais; o LaTeX aceita png e pdf)
ESCALA_PNG      = 3             # resolução do PNG: 1 = tela, 3 = ~300 dpi (bom para impressão)
PASTA_IMAGENS   = RAIZ / "analise" / "resultados"

# ---------- 12) RELATÓRIO HTML (etapa 11) ----------
GERAR_RELATORIO = True
ARQUIVO_RELATORIO = RAIZ / "analise" / "resultados" / "relatorio_quedas_bruscas.html"
TITULO_RELATORIO = "Quedas e aumentos bruscos de matrículas (INEP)"
EXIBIR_NO_NOTEBOOK = True       # False = não desenha nada no notebook (só monta o relatório; roda mais leve)
RELATORIO_ESTATICO = False      # False = gráficos interativos (tooltips; precisa de internet para carregar o Vega)
                                # True  = gráficos viram SVG fixos, arquivo 100% offline (precisa: pip install vl-convert-python)

## Etapa 2 — Carregar a base e calcular a variação % ano a ano

Para cada escola: `variação % = (matrículas no ano t − matrículas no ano t−1) / matrículas no ano t−1 × 100`.
Só entra a comparação se o ano anterior for **exatamente t−1** (escola que some e volta não gera comparação falsa).

In [138]:
def _ler(origem, ext):
    """Lê CSV/Excel/Parquet a partir de um caminho ou de bytes em memória."""
    if ext in (".xlsx", ".xls"):
        return pd.read_excel(origem)
    if ext == ".parquet":
        return pd.read_parquet(origem)
    return pd.read_csv(origem, sep=CSV_SEP, encoding=CSV_ENCODING, low_memory=False)


def carregar(caminho):
    p = Path(caminho)
    if p.suffix.lower() != ".zip":
        return _ler(p, p.suffix.lower())

    with zipfile.ZipFile(p) as z:
        validos = [n for n in z.namelist()
                   if not n.endswith("/") and not n.startswith("__MACOSX")
                   and Path(n).suffix.lower() in (".csv", ".txt", ".xlsx", ".xls", ".parquet")]
        validos.sort(key=lambda n: Path(n).suffix.lower() == ".txt")   # .csv/.xlsx/.parquet têm prioridade sobre .txt
        print("Arquivos no zip:", validos)
        if not validos:
            raise FileNotFoundError("Nenhum .csv/.xlsx/.parquet encontrado dentro do zip.")
        nome = NOME_ARQUIVO_NO_ZIP or validos[0]
        if nome not in z.namelist():
            raise FileNotFoundError(f"'{nome}' não está no zip. Opções: {validos}")
        if NOME_ARQUIVO_NO_ZIP is None and len(validos) > 1:
            print(f"Aviso: o zip tem {len(validos)} arquivos; lendo '{nome}'. Use NOME_ARQUIVO_NO_ZIP para escolher.")
        print("Lendo:", nome)
        dados_bytes = io.BytesIO(z.read(nome))
    ext = Path(nome).suffix.lower()
    return _ler(dados_bytes, ".csv" if ext == ".txt" else ext)


ETAPAS_VALIDAS = ["AI", "AF", "EM"]


def adicionar_rede(df):
    """Cria REDE a partir de DEP_ADM: Federal, Estadual e Municipal -> "Pública"; Privada -> "Privada"."""
    if "DEP_ADM" not in df.columns:
        return df
    dep = df["DEP_ADM"].astype("string").str.strip().str.lower()
    rede = pd.Series(pd.NA, index=df.index, dtype="object")
    rede[dep.isin(["federal", "estadual", "municipal", "pública", "publica"]).fillna(False)] = "Pública"
    rede[dep.eq("privada").fillna(False)] = "Privada"
    df = df.copy()
    df["REDE"] = rede
    return df


def etapas_da_selecao(etapa):
    """Interpreta ETAPA: "TOTAL", "AI", "AF", "EM" ou somas como "AI+AF" / "AF+EM" (qualquer ordem)."""
    if str(etapa).strip().upper() == "TOTAL":
        return list(ETAPAS_VALIDAS)
    partes = [p.strip().upper() for p in str(etapa).split("+")]
    invalidas = [p for p in partes if p not in ETAPAS_VALIDAS]
    if invalidas or len(set(partes)) != len(partes):
        raise ValueError(f"ETAPA inválida: {etapa!r}. Use 'TOTAL', 'AI', 'AF', 'EM' ou somas de duas "
                         f"etapas diferentes, como 'AI+AF', 'AF+EM' ou 'AI+EM'.")
    return [p for p in ETAPAS_VALIDAS if p in partes]       # ordem fixa: AI, AF, EM


def colunas_matricula(etapa, integral):
    etapas = etapas_da_selecao(etapa)
    cols = []
    for e in etapas:
        cols.append(f"QT_MAT_{e}_REG")
        if integral:
            cols.append(f"QT_MAT_{e}_INT")
    return cols


def preparar_base(raw, etapa, integral):
    """Devolve uma linha por escola/ano com MAT_ANT, MATRICULAS e VAR_PCT (já filtrada)."""
    cols_mat = colunas_matricula(etapa, integral)
    faltando = [c for c in ["ANO", "CO_ESCOLA"] + cols_mat if c not in raw.columns]
    if faltando:
        raise KeyError(f"Colunas ausentes na base: {faltando}")

    d = raw.copy()
    d["MATRICULAS"] = d[cols_mat].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)

    # escola duplicada no mesmo ano -> soma as matrículas, mantém o resto
    if d.duplicated(["CO_ESCOLA", "ANO"]).any():
        n = d.duplicated(["CO_ESCOLA", "ANO"]).sum()
        print(f"Aviso: {n} linhas duplicadas (escola, ano). Somando matrículas.")
        outras = {c: "first" for c in d.columns if c not in ("CO_ESCOLA", "ANO", "MATRICULAS")}
        d = d.groupby(["CO_ESCOLA", "ANO"], as_index=False).agg({**outras, "MATRICULAS": "sum"})

    d = d.sort_values(["CO_ESCOLA", "ANO"]).reset_index(drop=True)
    g = d.groupby("CO_ESCOLA")
    d["ANO_ANT"] = g["ANO"].shift(1)
    d["MAT_ANT"] = g["MATRICULAS"].shift(1)

    d = d[(d["ANO"] - d["ANO_ANT"]) == 1]                    # só anos consecutivos
    d = d[d["MAT_ANT"] >= max(MATRICULA_MINIMA_ANO_ANTERIOR, 1)]   # evita divisão por zero
    if EXCLUIR_FECHAMENTO:
        d = d[d["MATRICULAS"] > 0]

    d = d[(d["ANO"] >= ANO_INICIO) & (d["ANO"] <= ANO_FIM)]
    for col, filtro in [("DEP_ADM", FILTRO_DEP_ADM), ("REDE", FILTRO_REDE), ("LOCALIZACAO", FILTRO_LOCALIZACAO),
                        ("REGIAO", FILTRO_REGIAO), ("UF", FILTRO_UF)]:
        if filtro is not None and col in d.columns:
            d = d[d[col].isin(filtro)]

    d["VAR_PCT"] = (d["MATRICULAS"] - d["MAT_ANT"]) / d["MAT_ANT"] * 100
    return d.reset_index(drop=True)


_CACHE_BASE = {}


def _chave_base(etapa):
    return repr((etapa, INCLUIR_INTEGRAL, ANO_INICIO, ANO_FIM, FILTRO_DEP_ADM, FILTRO_REDE, FILTRO_LOCALIZACAO,
                 FILTRO_REGIAO, FILTRO_UF, MATRICULA_MINIMA_ANO_ANTERIOR, EXCLUIR_FECHAMENTO))


def obter_base(etapa):
    """preparar_base com cache: cenários que só mudam k/quartis não refazem a base."""
    chave = _chave_base(etapa)
    if chave not in _CACHE_BASE:
        if len(_CACHE_BASE) >= 6:
            _CACHE_BASE.pop(next(iter(_CACHE_BASE)))
        _CACHE_BASE[chave] = preparar_base(raw, etapa, INCLUIR_INTEGRAL)
    return _CACHE_BASE[chave]


raw = adicionar_rede(carregar(CAMINHO_ARQUIVO))
print("Base bruta:", raw.shape, "| anos:", raw["ANO"].min(), "a", raw["ANO"].max())

RELATORIO.clear()        # a cada execução a partir daqui, o relatório recomeça do zero
_CACHE_BASE.clear()
NOME_PRINCIPAL = "Principal · " + nome_cenario()
CENARIO_ATUAL = NOME_PRINCIPAL
secao("Parâmetros")
registrar_parametros()
secao("Etapa 2 — Base e variação % ano a ano")

base = obter_base(ETAPA)
print("Observações escola-ano usadas:", len(base))
print("Escolas distintas:", base["CO_ESCOLA"].nunique())

DIMS = [d for d in DIMENSOES if d in base.columns]
if ETAPA != "TOTAL" and f"ENS_PRED_{ETAPA}" in base.columns:
    DIMS.append(f"ENS_PRED_{ETAPA}")
print("Dimensões de cruzamento:", DIMS)

desc = base.groupby("ANO")["VAR_PCT"].describe(percentiles=[.05, .25, .5, .75, .95])
mostrar_tabela(desc, "Distribuição da variação % por ano")

Arquivos no zip: ['bd5.csv']
Lendo: bd5.csv
Base bruta: (2051631, 28) | anos: 2011 a 2025
Observações escola-ano usadas: 1501476
Escolas distintas: 136199
Dimensões de cruzamento: ['REDE', 'DEP_ADM', 'LOCALIZACAO', 'REGIAO', 'UF']


,count,mean,std,min,5%,25%,50%,75%,95%,max
ANO,,,,,,,,,,
2012,"111,664.00",-0.68,22.93,-99.50,-26.32,-9.62,-2.30,5.27,28.00,"1,082.86"
2013,"110,969.00",-0.61,23.55,-99.27,-26.36,-9.24,-2.13,5.41,27.43,"1,439.47"
2014,"110,159.00",-0.11,21.34,-99.33,-24.62,-8.53,-1.63,5.90,27.03,678.18
2015,"109,411.00",-0.81,20.56,-99.00,-24.32,-8.97,-2.12,5.08,25.00,968.75
2016,"108,775.00",-0.09,20.09,-99.07,-23.34,-7.99,-1.18,5.77,24.66,978.26
2017,"108,290.00",-0.30,20.46,-97.85,-23.81,-8.24,-1.50,5.41,25.00,"1,280.52"
2018,"107,645.00",-0.22,20.74,-98.52,-23.88,-8.04,-1.29,5.56,25.00,"1,229.03"
2019,"106,941.00",-0.65,20.36,-99.47,-23.53,-8.43,-1.67,4.94,23.68,"1,141.42"
2020,"106,198.00",-0.41,19.07,-98.05,-22.95,-7.61,-0.98,5.22,21.95,"1,241.18"


### Diagnóstico: onde as linhas se perdem em cada ano?
Se só aparecer 1 ano nos gráficos, esta tabela mostra o motivo:
- `linhas_brutas`: linhas do arquivo naquele ano;
- `com_matricula>0`: linhas com matrícula positiva na etapa escolhida;
- `com_ano_anterior`: escolas que também existem no ano t−1 (mesmo `CO_ESCOLA`);
- `usadas_na_analise`: o que sobrou depois de todos os filtros.

In [139]:
chaves = raw[["CO_ESCOLA", "ANO"]].drop_duplicates()
anteriores = chaves.assign(ANO=chaves["ANO"] + 1)
com_prev = chaves.merge(anteriores, on=["CO_ESCOLA", "ANO"]).groupby("ANO").size()

mat_raw = raw[colunas_matricula(ETAPA, INCLUIR_INTEGRAL)].apply(pd.to_numeric, errors="coerce").fillna(0).sum(axis=1)

diag = pd.DataFrame({
    "linhas_brutas": raw.groupby("ANO").size(),
    "com_matricula>0": raw[mat_raw > 0].groupby("ANO").size(),
    "com_ano_anterior": com_prev,
    "usadas_na_analise": base.groupby("ANO").size(),
}).fillna(0).astype(int)
mostrar_tabela(diag, "Diagnóstico: onde as linhas se perdem em cada ano")

if base["ANO"].nunique() < 2:
    mostrar_texto("Só 1 ano chegou na análise. Veja a tabela acima: se 'linhas_brutas' tem vários anos mas "
                  "'com_ano_anterior' é 0, o CO_ESCOLA não bate entre anos; se 'com_matricula>0' é 0, "
                  "as colunas QT_MAT_* estão vazias nesses anos.", nivel="aviso")

,linhas_brutas,com_matricula>0,com_ano_anterior,usadas_na_analise
ANO,,,,
2011,151509,151503,0,0
2012,150305,150295,146436,111664
2013,147185,147185,143863,110969
2014,143958,143952,141316,110159
2015,140864,140856,138526,109411
2016,139813,139809,137116,108775
2017,137296,137291,134637,108290
2018,134454,134450,132086,107645
2019,132456,132455,130056,106941


## Etapa 3 — Limites de Tukey e marcação das quedas bruscas

- **Limite por ano:** Q1/Q3 calculados dentro de cada ano → "queda atípica *naquele ano*".
- **Limite global:** Q1/Q3 com todos os anos juntos → um único corte. Mostra o efeito de anos como 2020/2021.

Colunas criadas (True/False): `QUEDA_ANO`, `QUEDA_GLOBAL`, `AUMENTO_ANO`, `AUMENTO_GLOBAL` e `BRUSCA_ANO`, `BRUSCA_GLOBAL` (queda OU aumento).

In [140]:
def _atualizar_derivadas():
    """Recalcula as variáveis que dependem dos parâmetros (também usada pelos cenários da etapa 10)."""
    global PREFIXO, ROTULO, FLAG, DIMS
    PREFIXO = {"queda": "QUEDA", "aumento": "AUMENTO", "ambos": "BRUSCA"}[DIRECAO]
    ROTULO = {"queda": "queda brusca", "aumento": "aumento brusco",
              "ambos": "variação brusca (queda ou aumento)"}[DIRECAO]
    FLAG = f"{PREFIXO}_{LIMITE_PARA_CRUZAMENTO}"
    dims = [d for d in DIMENSOES if d in raw.columns]
    if ETAPA != "TOTAL" and f"ENS_PRED_{ETAPA}" in raw.columns:
        dims.append(f"ENS_PRED_{ETAPA}")
    DIMS = dims


_atualizar_derivadas()


def aplicar_limites(d, q_inf, q_sup, mult, criterio, corte_abs):
    d = d.copy()

    # --- por ano ---
    tab = d.groupby("ANO")["VAR_PCT"].agg(
        N="size",
        Q_INF=lambda s: s.quantile(q_inf),
        MEDIANA="median",
        Q_SUP=lambda s: s.quantile(q_sup),
    )
    tab["AIQ"] = tab["Q_SUP"] - tab["Q_INF"]
    tab["LIM_INF_ANO"] = tab["Q_INF"] - mult * tab["AIQ"]
    tab["LIM_SUP_ANO"] = tab["Q_SUP"] + mult * tab["AIQ"]
    d = d.join(tab[["LIM_INF_ANO", "LIM_SUP_ANO"]], on="ANO")

    # --- global ---
    s = d["VAR_PCT"]
    q1, q3 = s.quantile(q_inf), s.quantile(q_sup)
    aiq = q3 - q1
    glob = {"Q_INF": q1, "MEDIANA": s.median(), "Q_SUP": q3, "AIQ": aiq,
            "LIM_INF": q1 - mult * aiq, "LIM_SUP": q3 + mult * aiq}
    d["LIM_INF_GLOBAL"] = glob["LIM_INF"]
    d["LIM_SUP_GLOBAL"] = glob["LIM_SUP"]

    # --- flags ---
    if criterio not in ("estatistico", "absoluto", "ambos"):
        raise ValueError("CRITERIO deve ser 'estatistico', 'absoluto' ou 'ambos'")
    abs_q = d["VAR_PCT"] <= corte_abs
    abs_a = d["VAR_PCT"] >= CORTE_ABSOLUTO_AUMENTO_PCT
    for modo in ["ANO", "GLOBAL"]:
        est_q = d["VAR_PCT"] < d[f"LIM_INF_{modo}"]
        est_a = d["VAR_PCT"] > d[f"LIM_SUP_{modo}"]
        if criterio == "estatistico":
            q, a = est_q, est_a
        elif criterio == "absoluto":
            q, a = abs_q, abs_a
        else:
            q, a = est_q & abs_q, est_a & abs_a
        d[f"QUEDA_{modo}"] = q
        d[f"AUMENTO_{modo}"] = a
        d[f"BRUSCA_{modo}"] = q | a
    return d, tab, glob


def relatar_limites(tab, glob):
    mostrar_texto("Limites GLOBAIS: " + " | ".join(f"{k} = {v:.2f}" for k, v in glob.items()), "Limites globais")
    mostrar_tabela(tab.round(2), "Limites por ano (quartis, AIQ e cercas)")
    inalc = tab.index[tab["LIM_INF_ANO"] <= -100].tolist()
    if glob["LIM_INF"] <= -100:
        mostrar_texto("Limite inferior GLOBAL <= -100%: nenhuma escola pode ser outlier negativo por esse critério.",
                      nivel="aviso")
    if inalc:
        mostrar_texto(f"Anos com limite inferior <= -100% (inalcançável pelo critério estatístico): {inalc}. "
                      "Dica: use CRITERIO='absoluto' ou diminua MULTIPLICADOR.", nivel="aviso")


dados, tab_ano, lim_global = aplicar_limites(
    base, Q_INFERIOR, Q_SUPERIOR, MULTIPLICADOR, CRITERIO, CORTE_ABSOLUTO_PCT)

secao("Etapa 3 — Limites de Tukey e marcação")
relatar_limites(tab_ano, lim_global)

Limites GLOBAIS: Q_INF = -8.22 | MEDIANA = -1.37 | Q_SUP = 5.56 | AIQ = 13.77 | LIM_INF = -28.88 | LIM_SUP = 26.22


,N,Q_INF,MEDIANA,Q_SUP,AIQ,LIM_INF_ANO,LIM_SUP_ANO
ANO,,,,,,,
2012,111664,-9.62,-2.30,5.27,14.89,-31.95,27.61
2013,110969,-9.24,-2.13,5.41,14.65,-31.22,27.38
2014,110159,-8.53,-1.63,5.90,14.43,-30.18,27.55
2015,109411,-8.97,-2.12,5.08,14.05,-30.05,26.16
2016,108775,-7.99,-1.18,5.77,13.76,-28.62,26.41
2017,108290,-8.24,-1.50,5.41,13.64,-28.70,25.87
2018,107645,-8.04,-1.29,5.56,13.60,-28.43,25.95
2019,106941,-8.43,-1.67,4.94,13.37,-28.48,24.99
2020,106198,-7.61,-0.98,5.22,12.83,-26.86,24.47


## Etapa 4 — Boxplots por ano

- Caixa = `Q_INFERIOR` a `Q_SUPERIOR` (com 0.25/0.75 é o boxplot clássico). Traço branco = mediana.
- Bigodes = menor/maior valor **dentro** da cerca. Traço **vermelho** = limite inferior (abaixo = queda brusca). Traço **laranja** = limite superior (acima = aumento brusco).
- Pontos = conforme `MOSTRAR_PONTOS`.
- O eixo y é ampliado na região central (`ZOOM_Y`, `ZOOM_MARGEM`). Para ver as cercas e os outliers, use `ZOOM_Y = "cercas"`.

In [141]:
def estatisticas_box(d, grupo, modo):
    li, ls, flag = f"LIM_INF_{modo}", f"LIM_SUP_{modo}", f"{PREFIXO}_{modo}"
    linhas = []
    for chave, g in d.groupby(grupo):
        chave = chave if isinstance(chave, tuple) else (chave,)
        v = g["VAR_PCT"]
        dentro = v[(v >= g[li].iloc[0]) & (v <= g[ls].iloc[0])]
        linha = dict(zip(grupo, chave))
        linha.update({
            "q_inf": v.quantile(Q_INFERIOR), "mediana": v.median(), "q_sup": v.quantile(Q_SUPERIOR),
            "bigode_inf": dentro.min(), "bigode_sup": dentro.max(),
            "lim_inf": g[li].iloc[0], "lim_sup": g[ls].iloc[0], "n": len(v), "pct_marcada": g[flag].mean() * 100,
        })
        linhas.append(linha)
    return pd.DataFrame(linhas)


def dominio_y(d, stats=None):
    """Eixo y: fixo (Y_MIN/Y_MAX), zoom no 'meio' (caixas) ou nas 'cercas'."""
    if Y_MIN is not None and Y_MAX is not None:
        return [float(Y_MIN), float(Y_MAX)]
    if stats is not None:                                   # facetas: usa as caixas de cada categoria
        q_lo, q_hi = stats["q_inf"].min(), stats["q_sup"].max()
    else:
        q_lo = d.groupby("ANO")["VAR_PCT"].quantile(Q_INFERIOR).min()
        q_hi = d.groupby("ANO")["VAR_PCT"].quantile(Q_SUPERIOR).max()
    if ZOOM_Y == "cercas":
        lo = min(q_lo, d["LIM_INF_ANO"].min(), d["LIM_INF_GLOBAL"].min())
        hi = max(q_hi, d["LIM_SUP_ANO"].max(), d["LIM_SUP_GLOBAL"].max())
        pad = 0.10 * (hi - lo)
    else:
        lo, hi = q_lo, q_hi
        pad = ZOOM_MARGEM * max(hi - lo, 1e-6)
    return [float(max(-100, lo - pad)), float(hi + pad)]


def tamanho_caixa(n_x, largura):
    return int(max(8, min(70, largura / max(n_x, 1) * 0.5)))


def camadas_boxplot(stats, base_chart, x_enc, dom, tam, cor_por=None):
    escala = alt.Scale(domain=dom, clamp=False, nice=False)
    foco = alt.FieldOneOfPredicate(field="ANO", oneOf=[int(a) for a in ANOS_FOCO])
    if cor_por:   # facetas: uma cor por categoria
        cor_caixa = alt.Color(f"{cor_por}:N", legend=None, scale=alt.Scale(scheme="set2"))
    else:
        cor_caixa = alt.condition(foco, alt.value(COR_CAIXA_FOCO), alt.value(COR_CAIXA))
    tip = [alt.Tooltip("ANO:O", title="Ano"), alt.Tooltip("n:Q", title="escolas"),
           alt.Tooltip("q_inf:Q", format=".1f", title="quartil inferior"),
           alt.Tooltip("mediana:Q", format=".1f", title="mediana"),
           alt.Tooltip("q_sup:Q", format=".1f", title="quartil superior"),
           alt.Tooltip("lim_inf:Q", format=".1f", title="limite inferior"),
           alt.Tooltip("lim_sup:Q", format=".1f", title="limite superior"),
           alt.Tooltip("pct_marcada:Q", format=".1f", title="% marcadas")]
    zero = base_chart.mark_rule(color="#9CA3AF", strokeDash=[5, 5], clip=True).encode(y=alt.Y(datum=0, scale=escala))
    bigode = base_chart.mark_rule(color="#6B7280", strokeWidth=1.6, clip=True).encode(
        x_enc, y=alt.Y("bigode_inf:Q", scale=escala, title="Variação % de matrículas vs. ano anterior"), y2="bigode_sup:Q")
    tampa_i = base_chart.mark_tick(color="#6B7280", size=tam * 0.5, thickness=2, clip=True).encode(x_enc, y=alt.Y("bigode_inf:Q", scale=escala))
    tampa_s = base_chart.mark_tick(color="#6B7280", size=tam * 0.5, thickness=2, clip=True).encode(x_enc, y=alt.Y("bigode_sup:Q", scale=escala))
    caixa = base_chart.mark_bar(size=tam, cornerRadius=3, clip=True, stroke="#2F3E4E", strokeWidth=0.8, opacity=0.92).encode(
        x_enc, y=alt.Y("q_inf:Q", scale=escala), y2="q_sup:Q", color=cor_caixa, tooltip=tip)
    mediana = base_chart.mark_tick(color="white", size=tam, thickness=3, clip=True).encode(x_enc, y=alt.Y("mediana:Q", scale=escala))
    lim_i = base_chart.mark_tick(color=COR_QUEDA, size=tam * 1.4, thickness=3, clip=True).encode(x_enc, y=alt.Y("lim_inf:Q", scale=escala))
    lim_s = base_chart.mark_tick(color=COR_AUMENTO, size=tam * 1.4, thickness=3, clip=True).encode(x_enc, y=alt.Y("lim_sup:Q", scale=escala))
    return [zero, bigode, tampa_i, tampa_s, caixa, mediana, lim_i, lim_s]


LEGENDA_BOX = ("Caixa = quartis {qi:.0%}–{qs:.0%} · traço branco = mediana · bigodes = dados dentro da cerca · "
               "traço vermelho = limite inferior · traço laranja = limite superior · roxo = anos em foco (gráfico geral) / cor = categoria (facetas)")


def pontos_boxplot(d, modo):
    flag, li, ls = f"{PREFIXO}_{modo}", f"LIM_INF_{modo}", f"LIM_SUP_{modo}"
    if MOSTRAR_PONTOS == "nenhum":
        return None
    if MOSTRAR_PONTOS == "marcados":
        p = d[d[flag]]
    else:
        p = d[(d["VAR_PCT"] < d[li]) | (d["VAR_PCT"] > d[ls])]
    p = p.sample(frac=1, random_state=0).groupby("ANO").head(MAX_PONTOS_POR_ANO)
    cols = [c for c in ["ANO", "VAR_PCT", "CO_ESCOLA", "NO_ESCOLA", "DEP_ADM", "UF", "MAT_ANT", "MATRICULAS"] if c in p.columns]
    return p[cols]



def camadas_boxplot_horizontal(stats, dom, tam):
    """Boxplot deitado: anos no eixo y (de cima para baixo) e variação % no eixo x."""
    escala = alt.Scale(domain=dom, clamp=False, nice=False)
    y_enc = alt.Y("ANO:O", title="Ano")
    foco = alt.FieldOneOfPredicate(field="ANO", oneOf=[int(a) for a in ANOS_FOCO])
    cor_caixa = alt.condition(foco, alt.value(COR_CAIXA_FOCO), alt.value(COR_CAIXA))
    tip = [alt.Tooltip("ANO:O", title="Ano"), alt.Tooltip("n:Q", title="escolas"),
           alt.Tooltip("q_inf:Q", format=".1f", title="quartil inferior"),
           alt.Tooltip("mediana:Q", format=".1f", title="mediana"),
           alt.Tooltip("q_sup:Q", format=".1f", title="quartil superior"),
           alt.Tooltip("lim_inf:Q", format=".1f", title="limite inferior"),
           alt.Tooltip("lim_sup:Q", format=".1f", title="limite superior"),
           alt.Tooltip("pct_marcada:Q", format=".1f", title="% marcadas")]
    b = alt.Chart(stats)
    zero = b.mark_rule(color="#9CA3AF", strokeDash=[5, 5], clip=True).encode(x=alt.X(datum=0, scale=escala))
    bigode = b.mark_rule(color="#6B7280", strokeWidth=1.6, clip=True).encode(
        y_enc, x=alt.X("bigode_inf:Q", scale=escala, title="Variação % de matrículas vs. ano anterior"), x2="bigode_sup:Q")
    tampa_i = b.mark_tick(color="#6B7280", size=tam * 0.5, thickness=2, clip=True).encode(y_enc, x=alt.X("bigode_inf:Q", scale=escala))
    tampa_s = b.mark_tick(color="#6B7280", size=tam * 0.5, thickness=2, clip=True).encode(y_enc, x=alt.X("bigode_sup:Q", scale=escala))
    caixa = b.mark_bar(size=tam, cornerRadius=3, clip=True, stroke="#2F3E4E", strokeWidth=0.8, opacity=0.92).encode(
        y_enc, x=alt.X("q_inf:Q", scale=escala), x2="q_sup:Q", color=cor_caixa, tooltip=tip)
    mediana = b.mark_tick(color="white", size=tam, thickness=3, clip=True).encode(y_enc, x=alt.X("mediana:Q", scale=escala))
    lim_i = b.mark_tick(color=COR_QUEDA, size=tam * 1.4, thickness=3, clip=True).encode(y_enc, x=alt.X("lim_inf:Q", scale=escala))
    lim_s = b.mark_tick(color=COR_AUMENTO, size=tam * 1.4, thickness=3, clip=True).encode(y_enc, x=alt.X("lim_sup:Q", scale=escala))
    return [zero, bigode, tampa_i, tampa_s, caixa, mediana, lim_i, lim_s]


def boxplot_por_ano(d, modo, titulo):
    stats = estatisticas_box(d, ["ANO"], modo)
    horizontal = BOXPLOT_ANO_ORIENTACAO == "horizontal"
    if BOXPLOT_ANO_ESCALA is not None:
        dom = [float(BOXPLOT_ANO_ESCALA[0]), float(BOXPLOT_ANO_ESCALA[1])]
        txt_eixo = f"Eixo fixo: {dom[0]:.0f}% a {dom[1]:.0f}% (cercas = traços vermelho e laranja; o que passa do eixo fica fora da vista)"
    else:
        dom = dominio_y(d)
        txt_eixo = (f"Eixo ampliado: {dom[0]:.1f}% a {dom[1]:.1f}% (o que sai dessa faixa fica fora da vista; "
                    "os números continuam nas tabelas e tooltips)")
    larg, altu = (BOXPLOT_ANO_LARGURA, BOXPLOT_ANO_ALTURA) if horizontal else (LARGURA_GRAFICO, ALTURA_GRAFICO)
    escala = alt.Scale(domain=dom, clamp=False, nice=False)
    pts = pontos_boxplot(d, modo)
    cor_pts = alt.condition(alt.datum.VAR_PCT > 0, alt.value(COR_AUMENTO), alt.value(COR_QUEDA))
    jitter = "sqrt(-2*log(random()))*cos(2*PI*random())"

    if horizontal:
        tam = tamanho_caixa(len(stats), altu)
        camadas = camadas_boxplot_horizontal(stats, dom, tam)
        if pts is not None and len(pts):
            camadas.append(
                alt.Chart(pts).mark_circle(size=26, opacity=0.5, clip=True).encode(
                    y=alt.Y("ANO:O", title="Ano"), yOffset="jitter:Q",
                    x=alt.X("VAR_PCT:Q", scale=escala), color=cor_pts, tooltip=[c for c in pts.columns])
                .transform_calculate(jitter=jitter))
    else:
        tam = tamanho_caixa(len(stats), larg)
        x_enc = alt.X("ANO:O", title="Ano", axis=alt.Axis(labelAngle=0))
        camadas = camadas_boxplot(stats, alt.Chart(stats), x_enc, dom, tam)
        if pts is not None and len(pts):
            camadas.append(
                alt.Chart(pts).mark_circle(size=26, opacity=0.5, clip=True).encode(
                    x_enc, xOffset="jitter:Q", y=alt.Y("VAR_PCT:Q", scale=escala),
                    color=cor_pts, tooltip=[c for c in pts.columns])
                .transform_calculate(jitter=jitter))

    graf = alt.layer(*camadas).properties(
        width=larg, height=altu,
        title=alt.Title(titulo, subtitle=[LEGENDA_BOX.format(qi=Q_INFERIOR, qs=Q_SUPERIOR), txt_eixo]))
    return estilizar(graf)

def rotulo_params():
    return (f"quartis {Q_INFERIOR:.2f}/{Q_SUPERIOR:.2f}, k={MULTIPLICADOR}, critério={CRITERIO}, "
            f"etapa={ETAPA}, mín. {MATRICULA_MINIMA_ANO_ANTERIOR} alunos")


sub = rotulo_params()

secao("Etapa 4 — Boxplots por ano")
mostrar(boxplot_por_ano(dados, "ANO", f"Limite POR ANO — {sub}"),
        "Boxplot por ano — limite calculado dentro de cada ano")

alt.LayerChart(...)

In [142]:
mostrar(boxplot_por_ano(dados, "GLOBAL", f"Limite GLOBAL — {sub}"),
        "Boxplot por ano — limite global (todos os anos juntos)")

alt.LayerChart(...)

## Etapa 5 — Quantas escolas tiveram queda ou aumento brusco em cada ano?
O boxplot mostra a distribuição; esta etapa resume **proporção** (e contagem) de escolas marcadas.

In [143]:
def resumo_por_ano(d):
    """Tabela (contagem e %) e gráfico de escolas com queda/aumento brusco por ano."""
    resumo = pd.DataFrame({"escolas": d.groupby("ANO").size()})
    for tipo in ["QUEDA", "AUMENTO"]:
        for modo in ["ANO", "GLOBAL"]:
            resumo[f"n_{tipo}_{modo}"] = d.groupby("ANO")[f"{tipo}_{modo}"].sum()
            resumo[f"pct_{tipo}_{modo}"] = resumo[f"n_{tipo}_{modo}"] / resumo["escolas"] * 100

    pcts = [c for c in resumo.columns if c.startswith("pct_")]
    longo = resumo.reset_index().melt(id_vars="ANO", value_vars=pcts, var_name="serie", value_name="pct")
    longo["tipo"] = np.where(longo["serie"].str.contains("QUEDA"), "queda brusca", "aumento brusco")
    longo["limite"] = np.where(longo["serie"].str.endswith("_ANO"), "limite por ano", "limite global")

    graf = estilizar(alt.Chart(longo).mark_line(point=alt.OverlayMarkDef(size=90, filled=True), strokeWidth=3).encode(
        x=alt.X("ANO:O", title="Ano", axis=alt.Axis(labelAngle=0)), y=alt.Y("pct:Q", title="% de escolas"),
        color=alt.Color("tipo:N", title=None, scale=alt.Scale(domain=["queda brusca", "aumento brusco"], range=[COR_QUEDA, COR_AUMENTO])),
        strokeDash=alt.StrokeDash("limite:N", title=None),
        tooltip=["ANO:O", "tipo:N", "limite:N", alt.Tooltip("pct:Q", format=".2f")]
    ).properties(width=LARGURA_GRAFICO, height=int(ALTURA_GRAFICO * 0.8), title="Proporção de escolas com queda e aumento brusco por ano"))
    return resumo, graf


secao("Etapa 5 — Proporção de quedas e aumentos por ano")
resumo, graf_resumo = resumo_por_ano(dados)
mostrar_tabela(resumo.round(2), "Contagem e % de escolas marcadas por ano")
mostrar(graf_resumo, "Proporção de quedas e aumentos bruscos por ano")

,escolas,n_QUEDA_ANO,pct_QUEDA_ANO,n_QUEDA_GLOBAL,pct_QUEDA_GLOBAL,n_AUMENTO_ANO,pct_AUMENTO_ANO,n_AUMENTO_GLOBAL,pct_AUMENTO_GLOBAL
ANO,,,,,,,,,
2012,111664,3475,3.11,4500,4.03,5689,5.09,6097,5.46
2013,110969,3833,3.45,4576,4.12,5563,5.01,5942,5.35
2014,110159,3475,3.15,3878,3.52,5372,4.88,5810,5.27
2015,109411,3330,3.04,3700,3.38,5081,4.64,5063,4.63
2016,108775,3377,3.10,3331,3.06,4867,4.47,4916,4.52
2017,108290,3574,3.30,3531,3.26,5157,4.76,5039,4.65
2018,107645,3728,3.46,3607,3.35,5073,4.71,5005,4.65
2019,106941,3506,3.28,3386,3.17,4983,4.66,4628,4.33
2020,106198,3675,3.46,3103,2.92,4449,4.19,3999,3.77


alt.Chart(...)

## Etapa 6 — Cruzamento com rede, localização, região, UF e modalidade

Usa a flag escolhida em `LIMITE_PARA_CRUZAMENTO`. Sempre compara **proporção** dentro de cada categoria (não contagem).

In [144]:
FLAG = f"{PREFIXO}_{LIMITE_PARA_CRUZAMENTO}"
print("Flag usada nos cruzamentos:", FLAG, "->", ROTULO)


def proporcao_por(d, dim, flag):
    t = d.groupby(["ANO", dim]).agg(N=(flag, "size"), QUEDAS=(flag, "sum")).reset_index()
    t["PCT_QUEDA"] = t["QUEDAS"] / t["N"] * 100
    return t[t["N"] >= N_MIN_CATEGORIA]


def grafico_proporcao(t, dim):
    n_cat = t[dim].nunique()
    tip = ["ANO:O", f"{dim}:N", "N:Q", "QUEDAS:Q", alt.Tooltip("PCT_QUEDA:Q", format=".2f")]
    if n_cat <= 8:
        g = alt.Chart(t).mark_line(point=alt.OverlayMarkDef(size=70, filled=True), strokeWidth=2.5).encode(
            x=alt.X("ANO:O", title="Ano"), y=alt.Y("PCT_QUEDA:Q", title=f"% com {ROTULO}"),
            color=alt.Color(f"{dim}:N"), tooltip=tip).properties(width=LARGURA_GRAFICO, height=int(ALTURA_GRAFICO * 0.7))
    else:   # muitas categorias (ex.: UF) -> mapa de calor
        g = alt.Chart(t).mark_rect().encode(
            x=alt.X("ANO:O", title="Ano"), y=alt.Y(f"{dim}:N", title=dim),
            color=alt.Color("PCT_QUEDA:Q", scale=alt.Scale(scheme="orangered"), title="% marcadas"),
            tooltip=tip).properties(width=LARGURA_GRAFICO, height=max(300, 20 * n_cat))
    return estilizar(g.properties(title=f"% de escolas com {ROTULO} por {dim} (limite {LIMITE_PARA_CRUZAMENTO.lower()})"))


def proporcao_por_tipo(d, dim, modo):
    """% de escolas com QUEDA e com AUMENTO brusco, separados, por ano e categoria de `dim`."""
    partes = []
    for tipo, rotulo in (("QUEDA", "queda"), ("AUMENTO", "aumento")):
        col = f"{tipo}_{modo}"
        t = d.groupby(["ANO", dim]).agg(N=(col, "size"), MARCADAS=(col, "sum")).reset_index()
        t["PCT"] = t["MARCADAS"] / t["N"] * 100
        t["tipo"] = rotulo
        partes.append(t)
    t = pd.concat(partes, ignore_index=True)
    return t[t["N"] >= N_MIN_CATEGORIA]


def grafico_proporcao_tipos(t, dim):
    """Uma linha por categoria e tipo: cor = categoria; contínua = aumento, tracejada = queda."""
    tip = ["ANO:O", f"{dim}:N", "tipo:N", "N:Q", "MARCADAS:Q", alt.Tooltip("PCT:Q", format=".2f", title="% marcadas")]
    g = alt.Chart(t).mark_line(point=alt.OverlayMarkDef(size=70, filled=True), strokeWidth=2.5).encode(
        x=alt.X("ANO:O", title="Ano", axis=alt.Axis(labelAngle=0)),
        y=alt.Y("PCT:Q", title="% de escolas"),
        color=alt.Color(f"{dim}:N", title=dim),
        strokeDash=alt.StrokeDash("tipo:N", title="Tipo",
                                  scale=alt.Scale(domain=["aumento", "queda"], range=[[1, 0], [6, 4]]),
                                  legend=alt.Legend(symbolType="stroke", symbolStrokeColor="#374151", symbolStrokeWidth=2.5,
                                                     symbolSize=900, symbolFillColor="transparent")),
        tooltip=tip,
    ).properties(width=LARGURA_GRAFICO, height=int(ALTURA_GRAFICO * 0.7))
    return estilizar(g.properties(
        title=f"% de escolas com queda (tracejada) e aumento (contínua) brusco por {dim} "
              f"(limite {LIMITE_PARA_CRUZAMENTO.lower()})"))


def relatar_cruzamentos(d):
    for dim in DIMS:
        # DIRECAO = "ambos": queda e aumento em linhas separadas (só para dimensões com poucas categorias;
        # dimensões com muitas categorias, como UF, continuam em mapa de calor com a marcação combinada).
        if DIRECAO == "ambos" and d[dim].nunique() <= 8:
            t = proporcao_por_tipo(d, dim, LIMITE_PARA_CRUZAMENTO)
            if t.empty:
                mostrar_texto(f"[{dim}] nenhuma categoria com pelo menos {N_MIN_CATEGORIA} escolas/ano.", nivel="aviso")
                continue
            mostrar(grafico_proporcao_tipos(t, dim), f"% de escolas com queda e com aumento brusco por {dim}")
            continue
        t = proporcao_por(d, dim, FLAG)
        if t.empty:
            mostrar_texto(f"[{dim}] nenhuma categoria com pelo menos {N_MIN_CATEGORIA} escolas/ano.", nivel="aviso")
            continue
        mostrar(grafico_proporcao(t, dim), f"% de escolas com {ROTULO} por {dim}")


secao("Etapa 6 — Cruzamentos")
relatar_cruzamentos(dados)

Flag usada nos cruzamentos: BRUSCA_ANO -> variação brusca (queda ou aumento)


alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

alt.Chart(...)

### 6.1 — Tabela de pandemia: % de escolas marcadas (conforme `DIRECAO`) por rede nos anos-foco

In [145]:
def relatar_pandemia(d):
    if "DEP_ADM" in d.columns:
        tp = (d[d["ANO"].isin(ANOS_FOCO)]
              .groupby(["DEP_ADM", "ANO"])[FLAG].mean().mul(100).unstack("ANO"))
        mostrar_tabela(tp.round(2), f"% de escolas marcadas ({ROTULO}) por rede nos anos {ANOS_FOCO}")
    if "REDE" in d.columns:
        tr = (d[d["ANO"].isin(ANOS_FOCO)]
              .groupby(["REDE", "ANO"])[FLAG].mean().mul(100).unstack("ANO"))
        mostrar_tabela(tr.round(2), f"% de escolas marcadas ({ROTULO}): pública × privada nos anos {ANOS_FOCO}")


secao("Etapa 6.1 — Tabela de pandemia")
relatar_pandemia(dados)

ANO,2019,2020,2021,2022
DEP_ADM,,,,
Estadual,5.79,6.20,5.63,7.27
Federal,17.41,12.58,7.33,9.15
Municipal,6.98,6.10,5.18,5.46
Privada,13.14,13.65,15.90,14.24


ANO,2019,2020,2021,2022
REDE,,,,
Privada,13.14,13.65,15.90,14.24
Pública,6.68,6.17,5.34,6.06


### 6.2 — Boxplots facetados por categoria
Mesmos limites da etapa 3 (calculados com todas as escolas do ano). Sem pontos individuais, para ficar leve.

In [146]:
def boxplot_facetado(d, dim, modo):
    dd = d[d["ANO"].isin(ANOS_FOCO)] if BOXPLOT_SO_ANOS_FOCO else d
    if dd.empty:
        print(f"[{dim}] Nenhuma linha nos anos {ANOS_FOCO}. Anos disponíveis: {sorted(int(a) for a in d['ANO'].unique())}")
        return None
    dd = dd.copy()
    dd[dim] = dd[dim].fillna("(vazio)").astype(str)   # NaN não derruba o groupby
    stats = estatisticas_box(dd, ["ANO", dim], modo)
    if stats.empty or "n" not in stats.columns:
        print(f"[{dim}] Sem dados para o boxplot.")
        return None
    stats = stats[stats["n"] >= N_MIN_CATEGORIA]
    if stats.empty:
        print(f"[{dim}] Nenhuma categoria com pelo menos {N_MIN_CATEGORIA} escolas/ano. Diminua N_MIN_CATEGORIA.")
        return None
    n_fac = stats[dim].nunique()
    n_anos = stats["ANO"].nunique()
    larg = int(max(110, (LARGURA_GRAFICO - 60 * n_fac) / n_fac))
    tam = tamanho_caixa(n_anos, larg)
    x_enc = alt.X("ANO:O", title="Ano", axis=alt.Axis(labelAngle=0))
    dom = dominio_y(d, stats)
    camadas = camadas_boxplot(stats, alt.Chart(stats), x_enc, dom, tam, cor_por=dim)
    graf = (alt.layer(*camadas).properties(width=larg, height=int(ALTURA_GRAFICO * 0.85))
            .facet(column=alt.Column(f"{dim}:N", title=None, header=alt.Header(labelFontSize=14, labelFontWeight="bold")))
            .properties(title=alt.Title(f"Boxplot por {dim} (limite {modo.lower()})",
                                        subtitle=[LEGENDA_BOX.format(qi=Q_INFERIOR, qs=Q_SUPERIOR), f"Eixo y ampliado: {dom[0]:.1f}% a {dom[1]:.1f}%"])))
    return estilizar(graf)


def relatar_facetados(d):
    for dim in BOXPLOT_POR:
        if dim in d.columns:
            graf = boxplot_facetado(d, dim, LIMITE_PARA_CRUZAMENTO)
            mostrar(graf, f"Boxplot facetado por {dim}")


secao("Etapa 6.2 — Boxplots facetados")
relatar_facetados(dados)

alt.FacetChart(...)

alt.FacetChart(...)

alt.FacetChart(...)

alt.FacetChart(...)

### 6.2b — Boxplots lado a lado em cada ano (ex.: rural × urbana)

Em cada ano, uma caixa por categoria da dimensão escolhida em `COMPARATIVOS` (por padrão `LOCALIZACAO` = rural × urbana).
- `COMPARATIVO_LIMITES = ["geral", "categoria"]` gera **dois gráficos**: um com as cercas de Tukey do conjunto de todas as escolas do ano (mesmo critério das outras etapas) e outro em que cada categoria tem **as suas próprias cercas**.
- Dá para comparar também `DEP_ADM` e `REGIAO`, restringir categorias (`COMPARATIVO_FILTRO`) e anos (`COMPARATIVO_ANOS`).
- Junto de cada gráfico saem duas tabelas: **mediana** e **% de escolas marcadas** por ano e categoria.

In [147]:
def _sem_marcas(d):
    """Remove colunas de limites/flags para poder recalcular com aplicar_limites()."""
    return d.drop(columns=[c for c in d.columns
                           if c.startswith(("LIM_INF_", "LIM_SUP_", "QUEDA_", "AUMENTO_", "BRUSCA_"))])


def boxplot_comparativo(d, dim, modo, limites="geral", anos=None, categorias=None):
    """Boxplots lado a lado: em cada ano, uma caixa por categoria de `dim`. Devolve (gráfico, estatísticas)."""
    dd = d.copy()
    dd[dim] = dd[dim].fillna("(vazio)").astype(str)
    if categorias:
        dd = dd[dd[dim].isin([str(c) for c in categorias])]
    if dd.empty:
        mostrar_texto(f"[{dim}] nenhuma linha para as categorias pedidas.", nivel="aviso")
        return None, None
    if limites == "categoria":      # cercas próprias de cada categoria
        dd = pd.concat([aplicar_limites(_sem_marcas(g), Q_INFERIOR, Q_SUPERIOR, MULTIPLICADOR, CRITERIO,
                                        CORTE_ABSOLUTO_PCT)[0] for _, g in dd.groupby(dim)])
    if anos:
        dd = dd[dd["ANO"].isin([int(a) for a in anos])]
    if dd.empty:
        mostrar_texto(f"[{dim}] nenhuma linha nos anos pedidos.", nivel="aviso")
        return None, None

    stats = estatisticas_box(dd, ["ANO", dim], modo)
    stats = stats[stats["n"] >= N_MIN_CATEGORIA].copy()
    if stats.empty:
        mostrar_texto(f"[{dim}] nenhuma categoria com pelo menos {N_MIN_CATEGORIA} escolas/ano. "
                      "Diminua N_MIN_CATEGORIA.", nivel="aviso")
        return None, None

    cats = sorted(stats[dim].unique())
    n_cat, n_anos = len(cats), stats["ANO"].nunique()
    dom = dominio_y(dd, stats)
    escala = alt.Scale(domain=dom, clamp=False, nice=False)
    tam = int(max(4, min(60, LARGURA_GRAFICO / max(n_anos, 1) * 0.8 / n_cat)))

    x = alt.X("ANO:O", title="Ano", axis=alt.Axis(labelAngle=0))
    off = alt.XOffset(f"{dim}:N", sort=cats)
    cor = alt.Color(f"{dim}:N", title=dim, scale=alt.Scale(domain=cats, scheme="set2"))
    tip = [alt.Tooltip("ANO:O", title="Ano"), alt.Tooltip(f"{dim}:N", title=dim), alt.Tooltip("n:Q", title="escolas"),
           alt.Tooltip("q_inf:Q", format=".1f", title="quartil inferior"),
           alt.Tooltip("mediana:Q", format=".1f", title="mediana"),
           alt.Tooltip("q_sup:Q", format=".1f", title="quartil superior"),
           alt.Tooltip("lim_inf:Q", format=".1f", title="limite inferior"),
           alt.Tooltip("lim_sup:Q", format=".1f", title="limite superior"),
           alt.Tooltip("pct_marcada:Q", format=".1f", title="% marcadas")]

    b = alt.Chart(stats)
    zero = b.mark_rule(color="#9CA3AF", strokeDash=[5, 5], clip=True).encode(y=alt.Y(datum=0, scale=escala))
    bigode = b.mark_rule(color="#6B7280", strokeWidth=1.6, clip=True).encode(
        x, xOffset=off, y=alt.Y("bigode_inf:Q", scale=escala, title="Variação % de matrículas vs. ano anterior"), y2="bigode_sup:Q")
    caixa = b.mark_bar(size=tam, cornerRadius=2, clip=True, stroke="#2F3E4E", strokeWidth=0.8, opacity=0.92).encode(
        x, xOffset=off, y=alt.Y("q_inf:Q", scale=escala), y2="q_sup:Q", color=cor, tooltip=tip)
    mediana = b.mark_tick(color="white", size=tam, thickness=3, clip=True).encode(
        x, xOffset=off, y=alt.Y("mediana:Q", scale=escala))
    lim_i = b.mark_tick(color=COR_QUEDA, size=tam, thickness=2.5, clip=True).encode(
        x, xOffset=off, y=alt.Y("lim_inf:Q", scale=escala))
    lim_s = b.mark_tick(color=COR_AUMENTO, size=tam, thickness=2.5, clip=True).encode(
        x, xOffset=off, y=alt.Y("lim_sup:Q", scale=escala))

    rot = "cercas gerais (todas as escolas do ano)" if limites == "geral" else "cercas próprias de cada categoria"
    legenda = (f"Caixa = quartis {Q_INFERIOR:.0%}–{Q_SUPERIOR:.0%} · traço branco = mediana · bigodes = dados dentro da cerca · "
               "traço vermelho/laranja = cerca inferior/superior · uma cor por categoria")
    graf = alt.layer(zero, bigode, caixa, mediana, lim_i, lim_s).properties(
        width=LARGURA_GRAFICO, height=int(ALTURA_GRAFICO * 0.85),
        title=alt.Title(f"Boxplot de {dim} em cada ano — {rot}",
                        subtitle=[legenda, f"{rotulo_params()} · eixo y ampliado: {dom[0]:.1f}% a {dom[1]:.1f}%"]))
    return estilizar(graf), stats


def relatar_comparativos(d):
    modo = LIMITE_PARA_CRUZAMENTO
    for dim in COMPARATIVOS:
        if dim not in d.columns:
            mostrar_texto(f"Comparativo ignorado: coluna {dim} não existe na base.", nivel="aviso")
            continue
        filtro = (COMPARATIVO_FILTRO or {}).get(dim)
        for i, lim in enumerate(COMPARATIVO_LIMITES):
            graf, stats = boxplot_comparativo(d, dim, modo, lim, COMPARATIVO_ANOS, filtro)
            if graf is None:
                continue
            rot = "cercas gerais" if lim == "geral" else "cercas próprias de cada categoria"
            mostrar(graf, f"{dim}: boxplots lado a lado em cada ano ({rot})")
            if i == 0:
                mostrar_tabela(stats.pivot(index="ANO", columns=dim, values="mediana").round(2),
                               f"{dim}: mediana da variação % por ano")
            mostrar_tabela(stats.pivot(index="ANO", columns=dim, values="pct_marcada").round(2),
                           f"{dim}: % de escolas marcadas por ano ({rot})")


secao("Etapa 6.2b — Comparativos lado a lado (rural × urbana etc.)")
relatar_comparativos(dados)

alt.LayerChart(...)

LOCALIZACAO,Rural,Urbana
ANO,,
2012,-4.47,-1.54
2013,-3.61,-1.60
2014,-2.78,-1.25
2015,-3.08,-1.75
2016,-2.82,-0.66
2017,-2.41,-1.21
2018,-2.78,-0.83
2019,-3.33,-1.12
2020,-3.23,-0.28


LOCALIZACAO,Rural,Urbana
ANO,,
2012,9.85,7.42
2013,10.71,7.45
2014,9.95,7.20
2015,9.36,6.98
2016,8.72,7.11
2017,10.17,7.21
2018,10.21,7.38
2019,9.44,7.37
2020,8.03,7.51


alt.LayerChart(...)

LOCALIZACAO,Rural,Urbana
ANO,,
2012,6.79,8.50
2013,7.20,8.70
2014,6.77,8.39
2015,6.38,7.96
2016,6.21,8.03
2017,6.91,8.38
2018,7.29,8.42
2019,6.89,8.21
2020,6.19,8.52


alt.LayerChart(...)

... (11 outputs left out)

REGIAO,Centro-Oeste,Nordeste,Norte,Sudeste,Sul
ANO,,,,,
2012,7.13,10.01,12.19,6.09,4.28
2013,7.42,11.26,11.77,5.89,3.41
2014,6.86,10.66,10.83,5.83,3.30
2015,7.38,9.85,9.93,5.77,3.82
2016,8.12,9.20,10.61,5.99,3.43
2017,8.42,10.44,10.87,5.64,4.03
2018,8.99,10.47,9.94,6.11,4.37
2019,7.96,10.07,10.51,6.01,3.98
2020,9.30,9.11,8.63,6.49,4.52


alt.LayerChart(...)

REGIAO,Centro-Oeste,Nordeste,Norte,Sudeste,Sul
ANO,,,,,
2012,7.56,7.33,8.13,8.48,6.96
2013,7.53,7.64,8.29,8.57,6.16
2014,8.34,7.54,7.50,7.89,6.42
2015,8.36,7.41,7.15,7.86,5.88
2016,8.75,6.82,7.29,7.69,6.09
2017,7.66,7.68,7.70,7.96,6.27
2018,8.84,7.99,7.51,8.11,6.23
2019,7.77,7.92,8.03,7.77,5.74
2020,7.70,7.11,6.67,8.69,6.52


⚠ Comparativo ignorado: coluna  não existe na base.


In [148]:
ETAPAS_BOX = ["AI", "AF", "EM"]   # etapas lado a lado (pode incluir "TOTAL")


def relatar_etapas_lado_a_lado():
    partes = []
    for et in ETAPAS_BOX:
        try:
            b = obter_base(et)
        except KeyError as e:
            mostrar_texto(f"Etapa {et} ignorada: {e}", nivel="aviso")
            continue
        dd, _, _ = aplicar_limites(b, Q_INFERIOR, Q_SUPERIOR, MULTIPLICADOR, CRITERIO, CORTE_ABSOLUTO_PCT)
        partes.append(dd.assign(ETAPA_ANALISADA=et))
    if not partes:
        return
    d = pd.concat(partes, ignore_index=True)
    graf, stats = boxplot_comparativo(d, "ETAPA_ANALISADA", LIMITE_PARA_CRUZAMENTO, "geral", COMPARATIVO_ANOS, None)
    if graf is None:
        return
    mostrar(graf, "Etapas lado a lado em cada ano (cercas de cada etapa)")
    mostrar_tabela(stats.pivot(index="ANO", columns="ETAPA_ANALISADA", values="mediana").round(2),
                   "Etapas: mediana da variação % por ano")
    mostrar_tabela(stats.pivot(index="ANO", columns="ETAPA_ANALISADA", values="pct_marcada").round(2),
                   "Etapas: % de escolas marcadas por ano")


secao("Etapa 6.2c — Etapas lado a lado")
relatar_etapas_lado_a_lado()

alt.LayerChart(...)

ETAPA_ANALISADA,AF,AI,EM
ANO,,,
2012,-2.35,-2.86,0.00
2013,-2.63,-2.22,-0.93
2014,-3.17,-1.06,0.00
2015,-2.57,-1.82,-2.49
2016,-1.27,-1.67,0.00
2017,-1.79,-1.27,-2.86
2018,-0.50,-1.56,-2.71
2019,-1.18,-1.69,-3.12
2020,0.00,-2.10,0.75


ETAPA_ANALISADA,AF,AI,EM
ANO,,,
2012,7.26,6.98,7.50
2013,7.59,7.48,7.06
2014,6.99,7.09,6.81
2015,7.03,7.09,6.05
2016,6.67,6.96,5.95
2017,6.78,7.42,6.50
2018,7.00,7.31,6.14
2019,6.79,7.18,6.02
2020,6.69,6.77,6.26


### 6.3 — Perfil das escolas marcadas
Compara a composição das escolas **marcadas (conforme `DIRECAO`)** com a composição de **todas** as escolas (nos anos de `PERFIL_ANOS`).
`razao > 1` → a categoria está sobrerrepresentada entre as marcadas.

In [149]:
def relatar_perfil(d):
    sub_p = d[d["ANO"].isin(PERFIL_ANOS)]
    if sub_p.empty:
        mostrar_texto(f"Nenhuma linha nos anos {PERFIL_ANOS} para montar o perfil.", nivel="aviso")
        return
    for dim in DIMS:
        comp = pd.DataFrame({
            "% das escolas (todas)": sub_p[dim].value_counts(normalize=True) * 100,
            "% das escolas marcadas": sub_p.loc[sub_p[FLAG], dim].value_counts(normalize=True) * 100,
            "n_marcadas": sub_p.loc[sub_p[FLAG], dim].value_counts(),
        }).fillna(0)
        comp["razao"] = comp["% das escolas marcadas"] / comp["% das escolas (todas)"]
        mostrar_tabela(comp.sort_values("razao", ascending=False).round(2), f"Perfil por {dim} (anos {PERFIL_ANOS})")


secao("Etapa 6.3 — Perfil das escolas marcadas")
relatar_perfil(dados)

,% das escolas (todas),% das escolas marcadas,n_marcadas,razao
REDE,,,,
Privada,19.63,36.43,11812,1.86
Pública,80.37,63.57,20614,0.79


,% das escolas (todas),% das escolas marcadas,n_marcadas,razao
DEP_ADM,,,,
Privada,19.63,36.43,11812,1.86
Federal,0.56,0.85,276,1.51
Estadual,25.64,20.80,6743,0.81
Municipal,54.16,41.93,13595,0.77


,% das escolas (todas),% das escolas marcadas,n_marcadas,razao
LOCALIZACAO,,,,
Rural,27.02,28.20,9143,1.04
Urbana,72.98,71.80,23283,0.98


,% das escolas (todas),% das escolas marcadas,n_marcadas,razao
REGIAO,,,,
Nordeste,36.66,43.77,14193,1.19
Centro-Oeste,6.57,7.62,2471,1.16
Norte,12.39,14.31,4639,1.15
Sudeste,31.30,26.27,8517,0.84
Sul,13.07,8.04,2606,0.62


,% das escolas (todas),% das escolas marcadas,n_marcadas,razao
UF,,,,
PB,2.77,3.98,1290,1.44
RR,0.41,0.57,185,1.39
PI,2.62,3.43,1112,1.31
BA,9.46,12.14,3936,1.28
MT,1.70,2.17,705,1.28
MA,6.09,7.54,2444,1.24
AL,1.85,2.29,742,1.24
AC,0.74,0.90,292,1.22
GO,3.01,3.64,1180,1.21


## Etapa 7 — Sensibilidade: quartis × multiplicadores
Refaz a classificação para todas as combinações de `GRADE_QUANTIS` × `GRADE_MULTIPLICADORES`
e mostra como a **% de escolas marcadas** (conforme `DIRECAO`) muda em cada ano.

In [150]:
def sensibilidade(d, grade_q, grade_m, criterio, corte):
    linhas = []
    for qi, qs in grade_q:
        for m in grade_m:
            dd, _, _ = aplicar_limites(d, qi, qs, m, criterio, corte)
            for modo in ["ANO", "GLOBAL"]:
                r = dd.groupby("ANO")[f"{PREFIXO}_{modo}"].mean() * 100
                for ano, val in r.items():
                    linhas.append({"quantis": f"{qi:.2f}/{qs:.2f}", "k": m,
                                   "limite": "por ano" if modo == "ANO" else "global",
                                   "ANO": ano, "pct": val})
    return pd.DataFrame(linhas)


sens = sensibilidade(base, GRADE_QUANTIS, GRADE_MULTIPLICADORES, CRITERIO, CORTE_ABSOLUTO_PCT)

graf_sens = estilizar(alt.Chart(sens).mark_line(point=True, strokeWidth=2.2).encode(
    x=alt.X("ANO:O", title="Ano"), y=alt.Y("pct:Q", title=f"% de escolas com {ROTULO}"),
    color=alt.Color("k:O", title="multiplicador k", scale=alt.Scale(scheme="viridis")),
    tooltip=["quantis", "k", "limite", "ANO:O", alt.Tooltip("pct:Q", format=".2f")]
).properties(width=int(LARGURA_GRAFICO / 2.4), height=220).facet(
    row=alt.Row("quantis:N", title="quartis (inf/sup)"), column=alt.Column("limite:N", title=None)
).properties(title="Sensibilidade do critério"))

secao("Etapa 7 — Sensibilidade")
mostrar(graf_sens, "Sensibilidade: % de escolas marcadas por quartis × k")

alt.FacetChart(...)

In [151]:
# Resumo: % médio de escolas marcadas (todos os anos) para cada combinação
resumo_sens = (sens.groupby(["limite", "quantis", "k"])["pct"].mean().unstack("k"))
mostrar_tabela(resumo_sens.round(2), "% médio de escolas marcadas (todos os anos) por quartis × k")

k                  1.00  1.50  2.00  2.50  3.00
limite  quantis                                
global  0.05/0.95  0.87  0.40  0.25  0.17  0.12
        0.10/0.90  2.46  1.23  0.70  0.43  0.30
        0.25/0.75 13.26  7.88  5.07  3.47  2.46
por ano 0.05/0.95  0.86  0.40  0.25  0.17  0.12
        0.10/0.90  2.46  1.23  0.69  0.43  0.30
        0.25/0.75 13.26  7.89  5.08  3.47  2.46

## Etapa 8 — Comparação entre etapas (AI, AF, EM, total)
Roda todo o pipeline para cada etapa de `ETAPAS_COMPARAR`, com os mesmos parâmetros das etapas anteriores.

In [152]:
def comparar_etapas():
    saida = []
    for et in ETAPAS_COMPARAR:
        try:
            b = obter_base(et)
            dd, _, _ = aplicar_limites(b, Q_INFERIOR, Q_SUPERIOR, MULTIPLICADOR, CRITERIO, CORTE_ABSOLUTO_PCT)
            r = dd.groupby("ANO")[FLAG].mean().mul(100).rename("pct").reset_index()
            r["etapa"] = et
            saida.append(r)
        except KeyError as e:
            mostrar_texto(f"Etapa {et} ignorada: {e}", nivel="aviso")
    return pd.concat(saida) if saida else None


def relatar_etapas():
    comp = comparar_etapas()
    if comp is None:
        return
    mostrar(estilizar(alt.Chart(comp).mark_line(point=alt.OverlayMarkDef(size=70, filled=True), strokeWidth=2.5).encode(
        x=alt.X("ANO:O", title="Ano"), y=alt.Y("pct:Q", title=f"% com {ROTULO}"),
        color="etapa:N", tooltip=["etapa", "ANO:O", alt.Tooltip("pct:Q", format=".2f")]
    ).properties(width=LARGURA_GRAFICO, height=int(ALTURA_GRAFICO * 0.7),
                 title=f"% de escolas com {ROTULO} por etapa (limite {LIMITE_PARA_CRUZAMENTO.lower()})")),
        "% de escolas marcadas por etapa")
    mostrar_tabela(comp.pivot(index="ANO", columns="etapa", values="pct").round(2), "% de escolas marcadas por etapa")


secao("Etapa 8 — Comparação entre etapas")
relatar_etapas()

alt.Chart(...)

etapa,AF,AI,EM,TOTAL
ANO,,,,
2012,7.26,6.98,7.50,8.21
2013,7.59,7.48,7.06,8.47
2014,6.99,7.09,6.81,8.03
2015,7.03,7.09,6.05,7.69
2016,6.67,6.96,5.95,7.58
2017,6.78,7.42,6.50,8.06
2018,7.00,7.31,6.14,8.18
2019,6.79,7.18,6.02,7.94
2020,6.69,6.77,6.26,7.65


## Etapa 8b — Queda × aumento brusco por etapa, um gráfico embaixo do outro

Mesmo gráfico da Etapa 6 (cor = categoria de `DIM_EMPILHADO`; **linha contínua = aumento**, **tracejada = queda**),
repetido para cada etapa de `ETAPAS_EMPILHADAS` e empilhado verticalmente. Cada etapa tem seus próprios limites de Tukey,
calculados só com as escolas daquela etapa. Os demais parâmetros (quartis, k, critério, filtros, limite ano/global) são os da célula de parâmetros.

In [158]:
ROTULO_ETAPA = {"AI": "Anos iniciais (AI)", "AF": "Anos finais (AF)", "EM": "Ensino médio (EM)", "TOTAL": "Total (AI + AF + EM)"}


def tabela_tipos_por_etapa(etapas, dim, modo):
    """% de escolas com queda e com aumento brusco por ano e categoria, para cada etapa."""
    partes = []
    for et in etapas:
        try:
            b = obter_base(et)
            dd, _, _ = aplicar_limites(b, Q_INFERIOR, Q_SUPERIOR, MULTIPLICADOR, CRITERIO, CORTE_ABSOLUTO_PCT)
            if dim not in dd.columns:
                raise KeyError(f"dimensão {dim!r} não existe na base")
            t = proporcao_por_tipo(dd, dim, modo)
        except (KeyError, ValueError) as e:
            mostrar_texto(f"Etapa {et} ignorada: {e}", nivel="aviso")
            continue
        t["etapa"] = et
        partes.append(t)
    return pd.concat(partes, ignore_index=True) if partes else None


def grafico_tipos_empilhado(t, dim, etapas):
    ymax = float(np.ceil(t["PCT"].max() + 0.5))
    escala_y = alt.Scale(domain=[0, ymax]) if EIXO_Y_COMPARTILHADO else alt.Scale(zero=True)
    tip = ["etapa:N", "ANO:O", f"{dim}:N", "tipo:N", "N:Q", "MARCADAS:Q",
           alt.Tooltip("PCT:Q", format=".2f", title="% marcadas")]
    paineis = []
    categorias = sorted(t[dim].unique(), key=lambda c: c != "Pública")   # Pública primeiro = linha contínua
    tracos = [[1, 0], [6, 4], [2, 2], [8, 3]][:len(categorias)]
    for et in etapas:
        d_et = t[t["etapa"] == et]
        if d_et.empty:
            continue
        ultimo = et == [e for e in etapas if not t[t["etapa"] == e].empty][-1]
        g = alt.Chart(d_et).mark_line(point=alt.OverlayMarkDef(size=60, filled=True), strokeWidth=2.5).encode(
            x=alt.X("ANO:O", title="Ano" if ultimo else None, axis=alt.Axis(labelAngle=0)),
            y=alt.Y("PCT:Q", title="% de escolas", scale=escala_y),
            color=alt.Color("tipo:N", title="Tipo",
                            scale=alt.Scale(domain=["aumento", "queda"], range=["#F39200", "#D7001D"])),
            strokeDash=alt.StrokeDash(f"{dim}:N", title=dim,
                                      scale=alt.Scale(domain=categorias, range=tracos),
                                      legend=alt.Legend(symbolType="stroke", symbolStrokeColor="#374151",
                                                         symbolStrokeWidth=2.5, symbolSize=900,
                                                         symbolFillColor="transparent")),
            tooltip=tip,
        ).properties(width=LARGURA_GRAFICO, height=ALTURA_PAINEL,
                     title=alt.TitleParams(ROTULO_ETAPA.get(et, et), fontSize=14, anchor="start"))
        paineis.append(g)
    graf = alt.vconcat(*paineis, spacing=18).properties(
        title=f"% de escolas com queda (tracejada) e aumento (contínua) brusco por {dim} e por etapa "
              f"(limite {LIMITE_PARA_CRUZAMENTO.lower()})")
    return estilizar(graf)


secao("Etapa 8b — Queda × aumento por etapa (empilhado)")
t_emp = tabela_tipos_por_etapa(ETAPAS_EMPILHADAS, DIM_EMPILHADO, LIMITE_PARA_CRUZAMENTO)
if t_emp is None or t_emp.empty:
    mostrar_texto("Nenhum dado para o gráfico empilhado (confira ETAPAS_EMPILHADAS e DIM_EMPILHADO).", nivel="aviso")
else:
    if t_emp[DIM_EMPILHADO].nunique() > 8:
        mostrar_texto(f"{DIM_EMPILHADO} tem mais de 8 categorias; o gráfico pode ficar ilegível.", nivel="aviso")
    mostrar(grafico_tipos_empilhado(t_emp, DIM_EMPILHADO, ETAPAS_EMPILHADAS),
            f"% de escolas com queda e com aumento brusco por {DIM_EMPILHADO}, por etapa")
    mostrar_tabela(t_emp.pivot_table(index=["etapa", "ANO"], columns=[DIM_EMPILHADO, "tipo"], values="PCT").round(2),
                   "% de escolas com queda e aumento brusco, por etapa", exibir=False)


alt.VConcatChart(...)

## Etapa 9 — Exportação e lista das escolas

In [154]:
colunas_saida = [c for c in ["ANO", "CO_ESCOLA", "NO_ESCOLA", "UF", "NO_MUNICIPIO", "REGIAO", "LOCALIZACAO",
                             "DEP_ADM", "MAT_ANT", "MATRICULAS", "VAR_PCT", "QUEDA_ANO", "QUEDA_GLOBAL", "AUMENTO_ANO", "AUMENTO_GLOBAL"]
                 if c in dados.columns]


def lista_marcadas(d):
    cols = [c for c in colunas_saida if c in d.columns]
    return d.loc[d[FLAG], cols].sort_values("VAR_PCT", key=lambda x: -x.abs())


quedas = lista_marcadas(dados)
secao("Etapa 9 — Lista das escolas marcadas")
mostrar_texto(f"{len(quedas)} observações escola-ano com {ROTULO} ({FLAG}), ordenadas pela maior variação absoluta.")
mostrar_tabela(quedas.head(20), f"As 20 maiores variações ({FLAG})")

if EXPORTAR_CSV:
    quedas.to_csv(ARQUIVO_SAIDA, index=False, encoding="utf-8-sig")
    print("Salvo em", ARQUIVO_SAIDA)

118458 observações escola-ano com variação brusca (queda ou aumento) (BRUSCA_ANO), ordenadas pela maior variação absoluta.


,ANO,CO_ESCOLA,NO_ESCOLA,UF,NO_MUNICIPIO,REGIAO,LOCALIZACAO,DEP_ADM,MAT_ANT,MATRICULAS,VAR_PCT,QUEDA_ANO,QUEDA_GLOBAL,AUMENTO_ANO,AUMENTO_GLOBAL
1121135,2022,35220224,MACHADO DE ASSIS EMEF,SP,Taboão da Serra,Sudeste,Urbana,Municipal,32.00,840,"2,525.00",False,False,True,True
947345,2025,33085455,COLEGIO CENECISTA CAPITAO LEMOS CUNHA,RJ,Rio de Janeiro,Sudeste,Urbana,Privada,38.00,903,"2,276.32",False,False,True,True
1014971,2025,35010214,COLEGIO ABACO UNIDADE IPIRANGA,SP,São Paulo,Sudeste,Urbana,Privada,40.00,828,"1,970.00",False,False,True,True
1023247,2022,35018508,LUIZ GONZAGA DA COSTA PROFESSOR,SP,Campinas,Sudeste,Urbana,Estadual,65.00,1100,"1,592.31",False,False,True,True
724719,2025,29436931,CENTRO TERRITORIAL DE EDUCACAO PROFISSIONAL RA...,BA,Jeremoabo,Nordeste,Rural,Estadual,69.00,1142,"1,555.07",False,False,True,True
1495863,2025,53008804,CED 02 DO CRUZEIRO,DF,Brasília,Centro-Oeste,Urbana,Estadual,30.00,479,"1,496.67",False,False,True,True
321357,2013,23010860,SAO BENEDITO EMEB,CE,São Benedito,Nordeste,Urbana,Municipal,38.00,585,"1,439.47",False,False,True,True
136876,2025,15180379,EMEF MOACIR FREITAS MACHADO,PA,Santarém,Norte,Urbana,Municipal,30.00,422,"1,306.67",False,False,True,True
880046,2017,32035527,EEEFM CLOVIS BORGES MIGUEL,ES,Serra,Sudeste,Urbana,Estadual,77.00,1063,"1,280.52",False,False,True,True
1071862,2023,35105429,PORTO UNIAO COLEGIO,SP,São Paulo,Sudeste,Urbana,Privada,35.00,476,"1,260.00",False,False,True,True


## Etapa 10 — Cenários extras (acumular parâmetros no mesmo relatório)

Roda **todo o pipeline** (etapas 3 a 6, 9 e, se quiser, 8) para cada cenário definido na seção **10) CENÁRIOS** dos parâmetros:
- `CENARIO_MULTIPLICADORES` × `CENARIO_QUANTIS` × `CENARIO_CRITERIOS` → todas as combinações (ex.: k=1,5 e k=2);
- `CENARIOS_EXTRA` → cenários livres (outra etapa, outro filtro, outra direção, outro corte absoluto...).

Cada cenário vira uma **aba** no relatório HTML, e uma aba **"Comparação entre cenários"** mostra lado a lado a % de escolas marcadas e as cercas por ano.

In [155]:
def resumo_cenario(d):
    """% de escolas com queda e com aumento brusco por ano (para comparar cenários)."""
    linhas = []
    for tipo, rotulo in [("QUEDA", "queda brusca"), ("AUMENTO", "aumento brusco")]:
        r = d.groupby("ANO")[f"{tipo}_{LIMITE_PARA_CRUZAMENTO}"].mean().mul(100)
        linhas.append(r.rename("pct").reset_index().assign(tipo=rotulo))
    return pd.concat(linhas, ignore_index=True)


def limites_cenario(tab):
    t = tab[["LIM_INF_ANO", "LIM_SUP_ANO"]].reset_index().melt(id_vars="ANO", var_name="cerca", value_name="valor")
    t["cerca"] = t["cerca"].map({"LIM_INF_ANO": "cerca inferior", "LIM_SUP_ANO": "cerca superior"})
    return t


@contextmanager
def cenario_ativo(cfg, nome=None):
    """Troca temporariamente os parâmetros de `cfg` e restaura tudo no fim."""
    global CENARIO_ATUAL, SECAO_ATUAL
    antigos = {k: globals()[k] for k in cfg}
    ant_cen, ant_sec = CENARIO_ATUAL, SECAO_ATUAL
    try:
        globals().update(cfg)
        _atualizar_derivadas()
        nome = nome or nome_cenario(cfg)
        usados = {x["cenario"] for x in RELATORIO}
        n = 2
        base_nome = nome
        while nome in usados:
            nome = f"{base_nome} ({n})"
            n += 1
        CENARIO_ATUAL = nome
        yield nome
    finally:
        globals().update(antigos)
        _atualizar_derivadas()
        CENARIO_ATUAL, SECAO_ATUAL = ant_cen, ant_sec


def rodar_cenario(cfg_in):
    cfg = {k: v for k, v in cfg_in.items() if k != "nome"}
    desconhecidos = [k for k in cfg if k not in globals()]
    if desconhecidos:
        raise KeyError(f"Parâmetro(s) desconhecido(s) no cenário: {desconhecidos}")
    t0 = time.time()
    with cenario_ativo(cfg, cfg_in.get("nome")) as nome:
        print(f"▶ Cenário: {nome}")
        secao("Parâmetros")
        registrar_parametros()
        d, tab, glob = aplicar_limites(obter_base(ETAPA), Q_INFERIOR, Q_SUPERIOR, MULTIPLICADOR, CRITERIO, CORTE_ABSOLUTO_PCT)

        secao("Etapa 3 — Limites de Tukey e marcação")
        relatar_limites(tab, glob)

        secao("Etapa 4 — Boxplots por ano")
        s = rotulo_params()
        mostrar(boxplot_por_ano(d, "ANO", f"Limite POR ANO — {s}"), "Boxplot por ano — limite calculado dentro de cada ano")
        mostrar(boxplot_por_ano(d, "GLOBAL", f"Limite GLOBAL — {s}"), "Boxplot por ano — limite global (todos os anos juntos)")

        secao("Etapa 5 — Proporção de quedas e aumentos por ano")
        resumo_, graf_ = resumo_por_ano(d)
        mostrar_tabela(resumo_.round(2), "Contagem e % de escolas marcadas por ano")
        mostrar(graf_, "Proporção de quedas e aumentos bruscos por ano")

        secao("Etapa 6 — Cruzamentos")
        relatar_cruzamentos(d)
        secao("Etapa 6.1 — Tabela de pandemia")
        relatar_pandemia(d)
        secao("Etapa 6.2 — Boxplots facetados")
        relatar_facetados(d)
        secao("Etapa 6.2b — Comparativos lado a lado (rural × urbana etc.)")
        relatar_comparativos(d)
        secao("Etapa 6.3 — Perfil das escolas marcadas")
        relatar_perfil(d)
        if CENARIOS_RODAR_ETAPAS:
            secao("Etapa 8 — Comparação entre etapas")
            relatar_etapas()
        secao("Etapa 9 — Lista das escolas marcadas")
        mostrar_tabela(lista_marcadas(d).head(20), f"As 20 maiores variações ({FLAG})")
        print(f"   concluído em {time.time() - t0:.0f}s")
        return nome, resumo_cenario(d), limites_cenario(tab)


def montar_cenarios():
    def chave(c):
        if c["CRITERIO"] == "absoluto":     # no critério absoluto, k e quartis não mudam nada: evita cenários repetidos
            return ("absoluto",)
        return (c["Q_INFERIOR"], c["Q_SUPERIOR"], float(c["MULTIPLICADOR"]), c["CRITERIO"])

    vistos = {chave(dict(Q_INFERIOR=Q_INFERIOR, Q_SUPERIOR=Q_SUPERIOR, MULTIPLICADOR=MULTIPLICADOR, CRITERIO=CRITERIO))}
    lista = []
    for k, (qi, qs), crit in product(CENARIO_MULTIPLICADORES, CENARIO_QUANTIS, CENARIO_CRITERIOS):
        cfg = dict(MULTIPLICADOR=k, Q_INFERIOR=qi, Q_SUPERIOR=qs, CRITERIO=crit)
        if chave(cfg) not in vistos:        # o que for igual ao Principal já foi rodado
            vistos.add(chave(cfg))
            lista.append(cfg)
    lista.extend(dict(c) for c in CENARIOS_EXTRA)
    return lista


cenarios = montar_cenarios()
print(f"Principal + {len(cenarios)} cenário(s) extra(s).")

comp_res = [resumo_cenario(dados).assign(cenario=NOME_PRINCIPAL)]
comp_lim = [limites_cenario(tab_ano).assign(cenario=NOME_PRINCIPAL)]
for cfg in cenarios:
    nome_c, r_, l_ = rodar_cenario(cfg)
    comp_res.append(r_.assign(cenario=nome_c))
    comp_lim.append(l_.assign(cenario=nome_c))

if len(comp_res) > 1:
    CENARIO_ATUAL = "Comparação entre cenários"
    res = pd.concat(comp_res, ignore_index=True)
    lim = pd.concat(comp_lim, ignore_index=True)

    secao("% de escolas marcadas por ano")
    graf = (alt.Chart(res).mark_line(point=alt.OverlayMarkDef(size=70, filled=True), strokeWidth=2.5).encode(
        x=alt.X("ANO:O", title="Ano", axis=alt.Axis(labelAngle=0)), y=alt.Y("pct:Q", title="% de escolas marcadas"),
        color=alt.Color("cenario:N", title="cenário", legend=alt.Legend(orient="bottom", columns=1)),
        tooltip=["cenario:N", "ANO:O", "tipo:N", alt.Tooltip("pct:Q", format=".2f")]
    ).properties(width=int(LARGURA_GRAFICO / 1.8), height=int(ALTURA_GRAFICO * 0.6))
        .facet(column=alt.Column("tipo:N", title=None))
        .properties(title=f"% de escolas com queda e com aumento brusco, por cenário (limite {LIMITE_PARA_CRUZAMENTO.lower()})"))
    mostrar(estilizar(graf), "Quedas e aumentos bruscos por ano — todos os cenários")
    for tipo in ["queda brusca", "aumento brusco"]:
        piv = res[res["tipo"] == tipo].pivot(index="ANO", columns="cenario", values="pct")
        mostrar_tabela(piv.round(2), f"% de escolas com {tipo} por ano, por cenário")

    secao("Cercas de Tukey por ano")
    graf = (alt.Chart(lim).mark_line(point=alt.OverlayMarkDef(size=60, filled=True), strokeWidth=2.2).encode(
        x=alt.X("ANO:O", title="Ano", axis=alt.Axis(labelAngle=0)), y=alt.Y("valor:Q", title="limite (variação %)"),
        color=alt.Color("cenario:N", title="cenário", legend=alt.Legend(orient="bottom", columns=1)),
        tooltip=["cenario:N", "ANO:O", "cerca:N", alt.Tooltip("valor:Q", format=".2f")]
    ).properties(width=int(LARGURA_GRAFICO / 1.8), height=int(ALTURA_GRAFICO * 0.6))
        .facet(column=alt.Column("cerca:N", title=None))
        .properties(title="Cercas de Tukey por ano, por cenário"))
    mostrar(estilizar(graf), "Cercas por ano — todos os cenários")
    for cerca in ["cerca inferior", "cerca superior"]:
        piv = lim[lim["cerca"] == cerca].pivot(index="ANO", columns="cenario", values="valor")
        mostrar_tabela(piv.round(2), f"{cerca.capitalize()} por ano, por cenário")

CENARIO_ATUAL = NOME_PRINCIPAL

Principal + 1 cenário(s) extra(s).
▶ Cenário: k=2 · quartis 0.25/0.75 · estatistico
Limites GLOBAIS: Q_INF = -8.22 | MEDIANA = -1.37 | Q_SUP = 5.56 | AIQ = 13.77 | LIM_INF = -35.77 | LIM_SUP = 33.11


,N,Q_INF,MEDIANA,Q_SUP,AIQ,LIM_INF_ANO,LIM_SUP_ANO
ANO,,,,,,,
2012,111664,-9.62,-2.30,5.27,14.89,-39.39,35.05
2013,110969,-9.24,-2.13,5.41,14.65,-38.54,34.70
2014,110159,-8.53,-1.63,5.90,14.43,-37.40,34.77
2015,109411,-8.97,-2.12,5.08,14.05,-37.07,33.19
2016,108775,-7.99,-1.18,5.77,13.76,-35.50,33.28
2017,108290,-8.24,-1.50,5.41,13.64,-35.52,32.69
2018,107645,-8.04,-1.29,5.56,13.60,-35.23,32.75
2019,106941,-8.43,-1.67,4.94,13.37,-35.17,31.68
2020,106198,-7.61,-0.98,5.22,12.83,-33.28,30.89


alt.LayerChart(...)

alt.LayerChart(...)

,escolas,n_QUEDA_ANO,pct_QUEDA_ANO,n_QUEDA_GLOBAL,pct_QUEDA_GLOBAL,n_AUMENTO_ANO,pct_AUMENTO_ANO,n_AUMENTO_GLOBAL,pct_AUMENTO_GLOBAL
ANO,,,,,,,,,
2012,111664,2044,1.83,2653,2.38,3991,3.57,4355,3.90
2013,110969,2323,2.09,2774,2.50,3813,3.44,4105,3.70
2014,110159,2095,1.90,2295,2.08,3661,3.32,3966,3.60
2015,109411,1933,1.77,2121,1.94,3465,3.17,3471,3.17
2016,108775,1972,1.81,1943,1.79,3290,3.02,3304,3.04
2017,108290,2143,1.98,2103,1.94,3530,3.26,3461,3.20
2018,107645,2284,2.12,2186,2.03,3521,3.27,3466,3.22
2019,106941,2117,1.98,2046,1.91,3490,3.26,3236,3.03
2020,106198,2193,2.07,1778,1.67,3013,2.84,2686,2.53


alt.Chart(...)

... (41 outputs left out)

cenario,Principal · k=1.5 · quartis 0.25/0.75 · estatistico,k=2 · quartis 0.25/0.75 · estatistico
ANO,,
2012,5.09,3.57
2013,5.01,3.44
2014,4.88,3.32
2015,4.64,3.17
2016,4.47,3.02
2017,4.76,3.26
2018,4.71,3.27
2019,4.66,3.26
2020,4.19,2.84


alt.FacetChart(...)

cenario,Principal · k=1.5 · quartis 0.25/0.75 · estatistico,k=2 · quartis 0.25/0.75 · estatistico
ANO,,
2012,-31.95,-39.39
2013,-31.22,-38.54
2014,-30.18,-37.40
2015,-30.05,-37.07
2016,-28.62,-35.50
2017,-28.70,-35.52
2018,-28.43,-35.23
2019,-28.48,-35.17
2020,-26.86,-33.28


cenario,Principal · k=1.5 · quartis 0.25/0.75 · estatistico,k=2 · quartis 0.25/0.75 · estatistico
ANO,,
2012,27.61,35.05
2013,27.38,34.70
2014,27.55,34.77
2015,26.16,33.19
2016,26.41,33.28
2017,25.87,32.69
2018,25.95,32.75
2019,24.99,31.68
2020,24.47,30.89


## Etapa 11 — Relatório HTML

Junta **todos os gráficos e tabelas** registrados ao longo do notebook (cenário principal + cenários extras + comparação) em um único arquivo HTML, com uma aba por cenário, índice por seção e seções que recolhem.
- Gráficos interativos (tooltip, botão de exportar PNG/SVG). Precisam de internet só para carregar o Vega; para um arquivo 100% offline use `RELATORIO_ESTATICO = True`.
- O arquivo é salvo na pasta do notebook (`ARQUIVO_RELATORIO`).

In [156]:
_CSS = """
:root{--bg:#F8FAFC;--card:#FFFFFF;--tx:#111827;--sub:#6B7280;--bd:#E5E7EB;--ac:#2F5D8A;--av:#FEF3C7;--avb:#F59E0B}
*{box-sizing:border-box}
body{margin:0;background:var(--bg);color:var(--tx);font:15px/1.5 -apple-system,"Segoe UI",Roboto,Helvetica,Arial,sans-serif}
header{background:#0F2A43;color:#fff;padding:28px 32px}
header h1{margin:0 0 6px;font-size:26px}
header p{margin:0;color:#CBD5E1;font-size:13px}
.abas{position:sticky;top:0;z-index:10;display:flex;gap:6px;flex-wrap:wrap;padding:10px 32px;background:#fff;border-bottom:1px solid var(--bd)}
.aba{border:1px solid var(--bd);background:#fff;color:var(--tx);border-radius:999px;padding:6px 14px;cursor:pointer;font-size:13px}
.aba:hover{border-color:var(--ac)}
.aba.ativa{background:var(--ac);border-color:var(--ac);color:#fff}
main{padding:20px 32px 60px;max-width:1400px;margin:0 auto}
.cenario>h2.cen{font-size:20px;margin:26px 0 8px;color:var(--ac);border-bottom:2px solid var(--ac);padding-bottom:6px}
.indice{display:flex;gap:8px;flex-wrap:wrap;margin:8px 0 14px}
.indice a{font-size:12px;color:var(--ac);text-decoration:none;border:1px solid var(--bd);background:#fff;border-radius:6px;padding:3px 9px}
.indice a:hover{background:#EEF4FA}
details.secao{background:var(--card);border:1px solid var(--bd);border-radius:10px;margin:12px 0;padding:0 16px}
details.secao>summary{cursor:pointer;font-weight:600;font-size:16px;padding:12px 0}
.bloco{margin:6px 0 22px}
.bloco h4{margin:0 0 6px;font-size:14px;color:#374151}
.nota{color:var(--sub);font-size:12px;margin:4px 0 0}
.vega,.svg,.tabela-wrap{overflow-x:auto;max-width:100%}
.tabela{border-collapse:collapse;font-size:12.5px;margin:0}
.tabela th,.tabela td{border-bottom:1px solid var(--bd);padding:4px 10px;text-align:right;white-space:nowrap}
.tabela thead th{background:#F3F4F6;position:sticky;top:0}
.tabela tbody th{text-align:left;background:#FAFAFA}
.aviso{background:var(--av);border-left:4px solid var(--avb);padding:8px 12px;border-radius:4px;font-size:13px;margin:8px 0}
.info{color:#374151;font-size:13px;margin:6px 0}
footer{color:var(--sub);font-size:12px;text-align:center;padding:20px}
@media print{.abas{display:none}details.secao{break-inside:avoid}header{background:#fff;color:#000}}
"""

_JS = """
(function () {
  var opcoes = {actions: {export: true, source: false, compiled: false, editor: false}, renderer: "canvas"};
  function desenhar(el) {
    if (el.dataset.ok) return;
    el.dataset.ok = "1";
    if (typeof vegaEmbed === "undefined") {
      el.innerHTML = '<p class="aviso">Não consegui carregar as bibliotecas Vega (sem internet?). Gere o relatório com RELATORIO_ESTATICO = True.</p>';
      return;
    }
    var spec = JSON.parse(document.getElementById(el.dataset.spec).textContent);
    vegaEmbed(el, spec, opcoes).catch(function (e) {
      el.innerHTML = '<p class="aviso">Erro ao desenhar o gráfico: ' + e + '</p>';
    });
  }
  var obs = new IntersectionObserver(function (entradas) {
    entradas.forEach(function (e) { if (e.isIntersecting) { desenhar(e.target); obs.unobserve(e.target); } });
  }, {rootMargin: "600px"});
  document.querySelectorAll(".vega").forEach(function (el) { obs.observe(el); });

  var abas = Array.prototype.slice.call(document.querySelectorAll(".aba"));
  var secs = Array.prototype.slice.call(document.querySelectorAll(".cenario"));
  function ir(i) {
    abas.forEach(function (a, j) { a.classList.toggle("ativa", i === j); });
    secs.forEach(function (s, j) { s.hidden = !(i === j || i === secs.length); });
    if (window.history && history.replaceState) history.replaceState(null, "", "#aba-" + i);
  }
  abas.forEach(function (a, i) { a.addEventListener("click", function () { ir(i); window.scrollTo(0, 0); }); });
  var m = /^#aba-(\\d+)$/.exec(location.hash);
  ir(m && +m[1] <= secs.length ? +m[1] : 0);
})();
"""


def _svg(spec):
    import vl_convert as vlc
    return vlc.vegalite_to_svg(json.dumps(spec, default=str))


def _cdn_vega(specs):
    """Scripts do Vega no CDN, na versão do Vega-Lite usada pelo Altair instalado."""
    major = 5
    for s in specs:
        achou = [p for p in str(s.get("$schema", "")).split("/") if p.startswith("v") and p[1:2].isdigit()]
        if achou:
            major = int(achou[-1][1:].split(".")[0])
            break
    vega, embed = (6, 7) if major >= 6 else (5, 6)
    base_url = "https://cdn.jsdelivr.net/npm"
    return (f'<script src="{base_url}/vega@{vega}"></script>\n'
            f'<script src="{base_url}/vega-lite@{major}"></script>\n'
            f'<script src="{base_url}/vega-embed@{embed}"></script>')


def gerar_relatorio_html(caminho, titulo, estatico=False):
    esc = _html.escape
    if not RELATORIO:
        print("Nada foi registrado ainda: rode as etapas anteriores.")
        return None

    cenarios_ord = []
    for it in RELATORIO:
        if it["cenario"] not in cenarios_ord:
            cenarios_ord.append(it["cenario"])
    cenarios_ord.sort(key=lambda c: not c.startswith("Comparação"))   # comparação em primeiro (ordem estável)

    specs_json, specs_raw, avisos = [], [], []
    secoes_html = []
    for ci, cen in enumerate(cenarios_ord):
        itens = [it for it in RELATORIO if it["cenario"] == cen]
        nomes_secao = []
        for it in itens:
            if it["secao"] not in nomes_secao:
                nomes_secao.append(it["secao"])
        indice = "".join(f'<a href="#s{ci}-{si}">{esc(s)}</a>' for si, s in enumerate(nomes_secao))
        partes = []
        for si, s in enumerate(nomes_secao):
            blocos = []
            for it in itens:
                if it["secao"] != s:
                    continue
                nota = f'<p class="nota">{esc(it["nota"])}</p>' if it.get("nota") else ""
                titulo_it = f'<h4>{esc(it["titulo"])}</h4>' if it.get("titulo") else ""
                if it["tipo"] == "grafico":
                    conteudo = None
                    if estatico:
                        try:
                            conteudo = f'<div class="svg">{_svg(it["spec"])}</div>'
                        except Exception as e:
                            avisos.append(f"SVG falhou em '{it['titulo']}': {e}")
                    if conteudo is None:
                        sid = len(specs_json)
                        specs_raw.append(it["spec"])
                        specs_json.append(json.dumps(it["spec"], default=str).replace("</", "<\\/"))
                        conteudo = f'<div class="vega" data-spec="spec-{sid}"></div>'
                    blocos.append(f'<div class="bloco">{titulo_it}{conteudo}{nota}</div>')
                elif it["tipo"] == "tabela":
                    blocos.append(f'<div class="bloco">{titulo_it}<div class="tabela-wrap">{it["html"]}</div>{nota}</div>')
                else:
                    classe = "aviso" if it.get("nivel") == "aviso" else "info"
                    blocos.append(f'<p class="{classe}">{esc(it["texto"])}</p>')
            partes.append(f'<details class="secao" id="s{ci}-{si}" open><summary>{esc(s)}</summary>{"".join(blocos)}</details>')
        secoes_html.append(f'<section class="cenario" id="aba{ci}"><h2 class="cen">{esc(cen)}</h2>'
                           f'<div class="indice">{indice}</div>{"".join(partes)}</section>')

    abas = "".join(f'<button class="aba" type="button">{esc(c)}</button>' for c in cenarios_ord)
    abas += '<button class="aba" type="button">Todos</button>'
    n_graf = sum(1 for it in RELATORIO if it["tipo"] == "grafico")
    n_tab = sum(1 for it in RELATORIO if it["tipo"] == "tabela")
    agora = datetime.datetime.now().strftime("%d/%m/%Y %H:%M")
    meta = (f"Gerado em {agora} · base: {esc(str(CAMINHO_ARQUIVO))} · {len(cenarios_ord)} aba(s) · "
            f"{n_graf} gráficos · {n_tab} tabelas")
    scripts_spec = "\n".join(f'<script type="application/json" id="spec-{i}">{s}</script>' for i, s in enumerate(specs_json))
    vega_js = _cdn_vega(specs_raw) if specs_json else ""

    doc = (f'<!DOCTYPE html>\n<html lang="pt-BR">\n<head>\n<meta charset="utf-8">\n'
           f'<meta name="viewport" content="width=device-width, initial-scale=1">\n<title>{esc(titulo)}</title>\n'
           f'<style>{_CSS}</style>\n{vega_js}\n</head>\n<body>\n'
           f'<header><h1>{esc(titulo)}</h1><p>{meta}</p></header>\n'
           f'<nav class="abas">{abas}</nav>\n<main>\n{"".join(secoes_html)}\n</main>\n'
           f'<footer>Relatório gerado pelo notebook de quedas bruscas.</footer>\n'
           f'{scripts_spec}\n<script>{_JS}</script>\n</body>\n</html>\n')

    saida = Path(caminho)
    saida.write_text(doc, encoding="utf-8")
    for a in avisos[:5]:
        print("⚠", a)
    print(f"Relatório salvo em: {saida.resolve()}  ({saida.stat().st_size / 1e6:.1f} MB, {n_graf} gráficos, {n_tab} tabelas)")
    return saida


if GERAR_RELATORIO:
    caminho_relatorio = gerar_relatorio_html(ARQUIVO_RELATORIO, TITULO_RELATORIO, RELATORIO_ESTATICO)

Relatório salvo em: C:\Users\niela\Documents\Trabalho Final - Inglês 2\analise\resultados\relatorio_quedas_bruscas.html  (4.0 MB, 46 gráficos, 57 tabelas)
